# 바이브 코딩으로 이해하는 공간분석: 히로시마 교통 접근성

이 노트북은 특정 도시의 결과를 외우는 자료가 아니라, **바이브 코딩으로 공간분석 기능을 탐색하고 데이터에서 인사이트를 도출하는 과정**을 보여주는 실습이다.

하나의 질문을 네 단계로 발전시킨다. (1) 파일과 좌표를 공간 데이터로 바꾸고, (2) 행정구역별 공급과 실제 커버리지를 비교하고, (3) 직선거리를 보행 네트워크 거리로 바꾸고, (4) 분석 결과를 인터랙티브 지도와 우선순위로 전달한다. 각 단계에서 “무엇을 계산할까?”보다 먼저 “어떤 공간분석 기능이 이 질문에 맞을까?”를 결정한다.

여기서 바이브 코딩은 코드를 무작정 자동 생성하는 방식이 아니다. 자연어로 분석 질문을 세우고, AI와 함께 데이터 구조·공간 연산·시각화 함수를 빠르게 시도한 뒤, 지도와 수치로 결과를 검증하는 반복 작업이다. 따라서 이 튜토리얼의 핵심 산출물은 지도 하나가 아니라 **재사용 가능한 분석 흐름과 검증 가능한 인사이트**다.

히로시마 데이터는 이 흐름을 설명하기 위한 사례다. 동일한 코드 패턴은 다른 도시의 행정구역·인구·정류장 데이터에도 적용할 수 있다.

---
### 발표 흐름: 질문 → 공간분석 기능 → 데이터 인사이트

| 단계 | 발표에서 던지는 질문 | 핵심 기능 | 얻는 인사이트 |
|---|---|---|---|
| 1 | 공간 데이터를 어떻게 분석 가능한 형태로 만들까? | GeoPandas, CRS, spatial join, 집계 | 어디에 시설이 몰리고 비어 있는가 |
| 2 | “정류장이 있다”와 “실제로 접근 가능하다”는 같은가? | buffer, overlay, uncovered geometry | 인구를 고려한 교통 사각지대는 어디인가 |
| 3 | 직선거리만으로 접근성을 판단해도 될까? | OSMnx 네트워크, shortest path | 실제 보행 경로에서 취약성이 어떻게 달라지는가 |
| 4 | 분석 결과를 어떻게 의사결정에 전달할까? | Folium/pydeck, ranking, interactive layer | 추가 시설 후보지를 어떤 기준으로 우선순위화할까 |

각 단계는 “코드 작성”으로 끝나지 않는다. 실행 결과를 지도·표·상식적 검토로 확인하고, 잘못된 가정이나 데이터 한계를 다음 분석 질문에 반영한다.

---
### Before we start / 시작하기 전

This is a 3-hour hands-on session -- everyone runs this notebook on
their own machine, not just watches. Before Lecture 1 starts, confirm:

1. `uv sync` has been run from the repo root (installs geopandas, osmnx,
   folium, lonboard, matplotlib, etc. into `.venv/`).
2. Your Jupyter kernel is the project's `.venv`, not a system Python --
   if imports below fail with `ModuleNotFoundError` even after `uv sync`,
   this is almost always why.
3. The repo was cloned with `data/hiroshima/*.gpkg` intact (they're small,
   committed files, not the large gitignored raw downloads).

Run the cell below -- it checks all three and tells you exactly what's
missing, if anything.

3시간 핸즈온 세션이다 -- 모두 자기 노트북을 직접 실행하며 따라온다, 구경만
하는 게 아니다. 1강 시작 전에 다음을 확인하자:

1. 저장소 루트에서 `uv sync`를 실행했는지 (geopandas, osmnx,
   folium, lonboard, matplotlib 등을 `.venv/`에 설치).
2. Jupyter 커널이 시스템 파이썬이 아니라 프로젝트의 `.venv`인지 --
   `uv sync` 후에도 아래 import가 `ModuleNotFoundError`로 실패한다면
   거의 항상 이것 때문이다.
3. `data/hiroshima/*.gpkg`가 온전한 상태로 clone되었는지 (용량 작은
   커밋된 파일들이지, gitignore된 대용량 원본 다운로드가 아니다).

아래 셀을 실행해보자 -- 세 가지를 모두 확인하고, 뭐가 빠졌는지 정확히
알려준다.
---

---
#### Troubleshooting / 문제 해결

Common issues, in the order they tend to show up:

- **`ModuleNotFoundError` even after `uv sync`** -- your Jupyter kernel is
  pointed at a system Python, not `.venv/`. Reselect the kernel.
- **Korean/Japanese text renders as tofu boxes in a matplotlib plot** --
  expected for plot titles/labels/legends, which are English-only in this
  notebook on purpose (see the font note in Lecture 1). Markdown prose
  isn't affected.
- **Lecture 3's `graph_from_polygon()` seems stuck** -- it's downloading
  from OpenStreetMap; give it up to ~2 minutes on slow wifi (see the note
  right before that cell).
- **Lecture 4's 3D map is blank / shows no basemap** -- either start the
  self-hosted PMTiles server from the break-time instructions above, or
  make sure you have internet for the Carto/Mapbox fallback; a fully
  offline machine with no local server will show the 3D blocks with no
  background map underneath.
- **Lecture 4's map opens with no error but nothing renders at all** --
  if you exported the HTML and opened it by double-clicking (`file://`),
  serve it over `http://` instead (`python -m http.server`) -- browsers
  block the map's background-worker scripts from a `file://` origin.

흔한 문제들을 발생 순서대로 정리했다:

- **`uv sync` 후에도 `ModuleNotFoundError`** -- Jupyter 커널이 `.venv/`가
  아니라 시스템 파이썬을 가리키고 있다. 커널을 다시 선택하자.
- **matplotlib 그래프에서 한글/일본어가 네모로 나옴** -- 그래프
  제목·축·범례는 의도적으로 영어만 쓴다(1강의 폰트 관련 설명 참고).
  마크다운 본문 텍스트에는 영향 없음.
- **3강 `graph_from_polygon()`이 멈춘 것 같음** -- OpenStreetMap에서
  다운로드 중이다. 느린 wifi에서는 최대 2분 정도 기다려보자(해당 셀
  직전의 안내 참고).
- **4강 3D 지도가 비어 있거나 배경지도가 안 보임** -- 위 휴식 시간 안내대로
  자체 호스팅 PMTiles 서버를 켜거나, Carto/Mapbox 폴백을 쓰려면 인터넷이
  필요하다. 로컬 서버도 인터넷도 없으면 3D 블록만 보이고 배경지도는
  안 보인다.
- **4강 지도를 열었는데 에러 없이 아무것도 안 보임** -- HTML을 내보내서
  더블클릭(`file://`)으로 열었다면 `http://`로 서빙해서 열자
  (`python -m http.server`) -- 브라우저가 `file://`에서는 지도의 백그라운드
  워커 스크립트 로드를 막는다.
---


In [ ]:
import importlib.util, os

REQUIRED_PACKAGES = ["pandas", "numpy", "matplotlib", "geopandas",
                     "folium", "osmnx", "networkx", "lonboard"]
missing_pkgs = [m for m in REQUIRED_PACKAGES if importlib.util.find_spec(m) is None]

_data_dir = os.path.join(os.getcwd(), "data", "hiroshima")
REQUIRED_FILES = [
    "hiroshima_city_admin.gpkg",
    "hiroshima_city_bus_stops.gpkg",
    "hiroshima_city_stations.gpkg",
]
missing_files = [f for f in REQUIRED_FILES if not os.path.exists(os.path.join(_data_dir, f))]

if missing_pkgs or missing_files:
    print("NOT READY / 준비 안 됨")
    if missing_pkgs:
        print("  missing packages / 누락된 패키지:", ", ".join(missing_pkgs))
        print("  -> run `uv sync` from the repo root, then select the project's .venv kernel")
        print("  -> 저장소 루트에서 `uv sync` 실행 후, 프로젝트 .venv 커널을 선택하세요")
    if missing_files:
        print("  missing data files / 누락된 데이터 파일:", ", ".join(missing_files))
        print("  -> confirm the repo was cloned with data/hiroshima/*.gpkg intact")
        print("  -> data/hiroshima/*.gpkg가 온전한지 확인하세요")
else:
    print("READY -- environment and data look good. / 준비 완료 -- 환경과 데이터 모두 정상.")

---
### GIS mini-glossary / GIS 미니 용어집

A quick reference for terms used throughout this notebook. Come back to
this if something later doesn't click.

이 노트북 전체에서 쓰는 용어를 미리 정리해둔다. 나중에 뭐가 헷갈리면
여기로 돌아오면 된다.

| Term | One-line meaning / 한줄 설명 |
|---|---|
| **CRS** (Coordinate Reference System / 좌표계) | How a location on Earth's curved surface maps to flat numbers. Two families: **geographic** (lat/lon degrees, e.g. EPSG:4326/WGS84) and **projected** (metres on a flat plane, e.g. EPSG:6670 for Hiroshima). Distance/area math needs a projected CRS; web maps need geographic. / 지구의 굽은 표면 위치를 평면 좌표로 바꾸는 방식. 지리좌표계(경위도, EPSG:4326/WGS84)와 투영좌표계(평면 위 미터, 히로시마는 EPSG:6670)로 크게 나눠진다. 거리·면적 계산엔 투영좌표계, 웹 지도엔 지리좌표계가 필요하다. |
| **EPSG code** | A number that names one specific CRS unambiguously (e.g. `4326` = WGS84). / 특정 좌표계 하나를 명확히 가리키는 번호(예: `4326` = WGS84). |
| **Chōme** (町丁目, 소지역) | Japan's finest standard statistical/administrative subdivision -- a town-block-level unit within a city, roughly comparable to a city district or neighbourhood-level unit elsewhere. This notebook's population and boundary data are published at chōme level; every "small area" mentioned throughout is one chōme. / 일본의 최소 표준 통계·행정 단위 -- 도시 내 마치(町) 블록 단위로, 다른 나라의 구·동 단위 행정구역과 대략 비슷한 개념이다. 이 노트북의 인구·경계 데이터는 chōme(소지역) 단위로 제공되며, 본문에서 "소지역"이라 부르는 모든 것이 chōme 하나를 가리킨다. |
| **GeoDataFrame** | A Pandas DataFrame with one extra `geometry` column holding shapes (Point/LineString/Polygon) instead of plain numbers. / 숫자 대신 도형(Point/LineString/Polygon)을 담는 `geometry` 컬럼이 하나 더 있는 Pandas DataFrame. |
| **Spatial join** (`sjoin`) | Like a normal table join, but the matching condition is spatial ("is this point inside that polygon?") instead of "do these ID columns match?" / 일반 테이블 조인과 비슷하지만, 매칭 조건이 "ID 컬럼이 같은가"가 아니라 "이 점이 저 폴리곤 안에 있는가" 같은 공간 조건이다. |
| **Buffer** | Grow a point/line/polygon outward by a fixed distance -- e.g. "everywhere within 300m of a bus stop." / 점·선·면을 일정 거리만큼 밖으로 확장 -- 예: "버스정류장 반경 300m 이내 전체". |
| **Union / Intersection / Difference** | Combine (∪), overlap-only (∩), or subtract (−) two shapes. Covered in depth in Lecture 2. / 두 도형을 합치거나(∪) 겹치는 부분만 남기거나(∩) 빼는(−) 연산. 2강에서 자세히 다룸. |
| **Dissolve** | Merge multiple rows that share a key into one row, unioning their geometry and aggregating their other columns (sum/first/etc). / 같은 키를 공유하는 여러 행을 한 행으로 병합 -- geometry는 union, 나머지 컬럼은 sum/first 등으로 집계. |
| **Areal weighting** (면적가중) | When a shape only partly overlaps a region of interest, count only that fraction of its value (population, etc) -- not the whole thing. The single most common source of "quietly wrong" numbers in spatial analysis. / 도형이 관심 영역에 일부만 걸치면 그 비율만큼만 값(인구 등)을 세는 것 -- 전체를 다 세지 않는다. 공간분석에서 "조용히 틀린" 수치의 가장 흔한 원인. |
| **Choropleth** | A map that colors each polygon by a data value (e.g. population density). / 데이터 값(인구밀도 등)에 따라 폴리곤을 색칠하는 지도. |
| **Isochrone** | The area reachable from a point within a fixed time/distance budget, following a real network -- not a straight-line circle. / 지점에서 일정 시간/거리 안에 실제 네트워크를 따라 도달 가능한 영역 -- 직선거리 원이 아니다. |
| **Node / Edge** | A network's two building blocks: a node is a point (intersection), an edge is a connection between two nodes (a road segment). Lecture 3 covers this in depth. / 네트워크의 두 구성요소: 노드는 점(교차로), 엣지는 노드 간 연결(도로 구간). 3강에서 자세히 다룸. |
| **Join key / unique key** (조인 키) | The column two tables are matched on. If it isn't actually unique in one of the tables, a join silently duplicates or drops rows -- the single most common bug pattern found and fixed in this project (Lecture 1's `region_id` dissolve is a direct example). / 두 테이블이 매칭되는 기준 컬럼. 한쪽에서 실제로 유일하지 않으면 조인이 행을 조용히 중복하거나 누락시킨다 -- 이 프로젝트에서 발견·수정된 가장 흔한 버그 유형(1강의 `region_id` dissolve가 직접적인 예).
---


### [Lecture 1] 공간 데이터를 숫자와 질문으로 바꾸기

바이브 코딩의 첫 단계는 라이브러리 이름을 나열하는 것이 아니라 분석 질문을 데이터 구조로 번역하는 일이다. 행정구역, 인구, 버스·철도 정류장 파일을 불러오고 GeoPandas로 geometry를 만든 뒤, CRS를 통일하고 spatial join으로 같은 공간 단위에 연결한다.

이 과정에서 확인할 것은 “코드가 실행되는가?”만이 아니다. 좌표 단위가 도(degree)인지 미터(metre)인지, 공간 조인이 기대한 행 수를 만드는지, 집계 결과가 지도에서 말이 되는지를 함께 검증한다. 그 결과로 정류장 수, 인구 대비 공급량, 구역별 분포를 계산하고 첫 번째 인사이트를 얻는다: 어디에 시설이 집중되고 어떤 지역이 상대적으로 비어 있는가.

### Pandas
- A data-analysis library for loading and organizing table-shaped (DataFrame) data efficiently
- The go-to tool for handling missing values, sorting, group aggregation, and merging
- Supports CSV, Excel, databases, and many other formats
- Best suited for general analysis of numeric and categorical data

### Pandas
- 표 형태(DataFrame) 데이터를 효율적으로 불러오고 정리하는 데이터 분석 라이브러리
- 결측치 처리, 정렬, 그룹 집계, 병합 등 기본 EDA 작업에 가장 많이 사용됨
- CSV·엑셀·데이터베이스 등 다양한 데이터 포맷을 지원
- 수치·범주형 데이터 중심의 일반 데이터 분석에 적합

### GeoPandas
- A spatial-data library that adds spatial information (geometry) on top of Pandas
- Reads GIS files like shp/geojson directly and can render them as maps
- Provides spatial operations: CRS conversion, spatial joins, buffers, area/distance calculations
- Specialized for administrative-area, transportation, and environmental spatial analysis

### GeoPandas
- Pandas에 공간 정보(geometry)를 결합한 공간 데이터 전용 라이브러리
- shp·geojson 등 GIS 파일을 바로 읽고 지도 형태로 시각화 가능
- 좌표계(CRS) 변환, 공간조인, 버퍼·면적·거리 계산 등 공간 연산 제공
- 행정구역, 교통, 환경 등 공간 기반 분석에 특화됨

In [ ]:
# Install the GeoPandas library
# GeoPandas 라이브러리 설치
#
# Not run here: this notebook runs inside a uv-managed environment
# (pyproject.toml / uv.lock) where geopandas is already installed. The
# venv has no pip binary, so %pip install would print a visible error
# box during a live run even though nothing is actually broken.
# 여기서는 실행하지 않음: 이 노트북은 uv로 관리되는 환경
# (pyproject.toml / uv.lock)에서 실행되며, geopandas는 이미 설치되어
# 있다. 이 venv에는 pip 자체가 없어 %pip install을 실행하면
# 실제로는 아무것도 깨지지 않았는데도 발표 중에 빨간 에러 박스가 떠서
# 무언가 잘못된 것처럼 보일 수 있다.
# %pip install geopandas


In [ ]:
import os
import warnings
from pathlib import Path

# Keep warnings visible: CRS, invalid-geometry, and API-deprecation warnings
# often identify real GIS correctness problems.
warnings.filterwarnings("default")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

In [ ]:
import matplotlib.font_manager as fm
import matplotlib as mpl

# A Japanese-capable font is required for place names (kanji/kana) to render
# in matplotlib. Yu Gothic ships with Windows 10/11; Hiragino Sans covers
# macOS. Adjust if running somewhere else.
# 지명(한자/가나)이 matplotlib에서 깨지지 않으려면 일본어를 지원하는 폰트가
# 필요하다. Yu Gothic은 Windows 10/11 기본 폰트, Hiragino Sans는 macOS용이다.
# 다른 환경에서 실행한다면 폰트를 바꿔야 할 수 있다.
try:
    mpl.rcParams["font.family"] = "Yu Gothic"       # Windows
except Exception:
    mpl.rcParams["font.family"] = "Hiragino Sans"   # Mac

mpl.rcParams["axes.unicode_minus"] = False

### 0. Folder setup / 0. 폴더 구조 설정

#### Data sources
- Administrative boundary + registered population (small-area statistics): e-Stat, https://www.e-stat.go.jp/gis
- Bus stop locations: KSJ (National Land Numerical Information) P11, https://nlftp.mlit.go.jp/ksj/gml/datalist/KsjTmplt-P11.html
- Rail / streetcar station locations: KSJ N02, https://nlftp.mlit.go.jp/ksj/gml/datalist/KsjTmplt-N02.html

#### 데이터 출처
- 행정구역 경계 + 등록인구(소지역 통계): e-Stat, https://www.e-stat.go.jp/gis
- 버스정류장 위치정보: 国土数値情報(KSJ) P11, https://nlftp.mlit.go.jp/ksj/gml/datalist/KsjTmplt-P11.html
- 철도/노면전차역 위치정보: 国土数値情報(KSJ) N02, https://nlftp.mlit.go.jp/ksj/gml/datalist/KsjTmplt-N02.html

#### Data licenses & attribution / 데이터 라이선스·출처 표기

- **e-Stat (小地域統計)**: published by Japan's Statistics Bureau under
  the Government of Japan Standard Terms of Use (Ver. 2.0), which is
  compatible with CC BY 4.0 -- free to use, reproduce, and adapt with
  attribution to the source.
- **KSJ / 国土数値情報 (bus stops P11, rail/streetcar N02)**: published by
  MLIT (国土交通省) under the same Standard Terms of Use.
- **OpenStreetMap** (Lecture 3's walking network, via OSMnx): (c)
  OpenStreetMap contributors, licensed under the Open Database License
  (ODbL) -- attribution is required wherever a map derived from this data
  is shown.

Suggested attribution line for slides/exports: "Data: e-Stat, MLIT
国土数値情報 (KSJ), (c) OpenStreetMap contributors."

- **e-Stat (소지역 통계)**: 일본 총무성 통계국이 "정부 표준 이용약관
  (2.0판)"으로 공개 -- CC BY 4.0과 호환되며, 출처를 표기하면 자유롭게
  이용·복제·가공할 수 있다.
- **KSJ / 국토수치정보 (버스정류장 P11, 철도/노면전차 N02)**: 국토교통성이
  동일한 "정부 표준 이용약관"으로 공개.
- **OpenStreetMap** (3강의 보행 네트워크, OSMnx 경유): (c) OpenStreetMap
  contributors, ODbL 라이선스 -- 이 데이터로 만든 지도를 보여줄 때는 출처
  표기가 필요하다.

슬라이드·내보내기용 출처 표기 예시: "Data: e-Stat, 国土交通省
国土数値情報(KSJ), (c) OpenStreetMap contributors." 


In [ ]:
# Locate the repository root even when Jupyter was launched from a subfolder.
_start = Path.cwd().resolve()
_candidates = [_start, *_start.parents]
BASE_DIR = next(
    (p for p in _candidates if (p / "data" / "hiroshima").is_dir()),
    None,
)
if BASE_DIR is None:
    raise FileNotFoundError(
        "Could not locate data/hiroshima. Start Jupyter inside the repository."
    )

DATA_DIR = BASE_DIR / "data" / "hiroshima"
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("repository root:", BASE_DIR)

### 1. Utility functions / 1. 유틸 함수 설정

In [ ]:
def read_csv_safely(path, **kwargs):
    """Read Japanese public CSV data without silently corrupting place names."""
    errors = []
    for encoding in ("utf-8-sig", "cp932", "shift_jis"):
        try:
            return pd.read_csv(path, encoding=encoding, **kwargs)
        except UnicodeDecodeError as exc:
            errors.append(f"{encoding}: {exc}")
    raise UnicodeError(
        f"Could not decode {path!s} with utf-8-sig, cp932, or shift_jis:\n"
        + "\n".join(errors)
    )

In [ ]:
def to_num(s):
    # Coerce a string/mixed column to numeric; anything unconvertible becomes NaN
    # 문자열이나 섞여 있는 값을 숫자형으로 변환하고, 변환할 수 없는 값은 NaN으로 처리
    return pd.to_numeric(s, errors="coerce")

In [ ]:
def norm_nm(x):
    if pd.isna(x):
        return None

    s = str(x).strip()
    s = s.replace("\u00A0", " ")
    s = " ".join(s.split())

    # Full-width digits/alphabet (１２３ＡＢＣ) show up inconsistently across
    # Japanese government open-data sources -- normalize to half-width so the
    # same place name matches across files.
    # 일본 공공데이터에는 전각 숫자/알파벳(１２３ＡＢＣ)이 출처마다 다르게
    # 섞여 나온다 -- 반각으로 통일해야 다른 파일 간 지명 매칭이 일치한다.
    zen = "０１２３４５６７８９ＡＢＣＤＥＦＧＨＩＪＫＬＭＮＯＰＱＲＳＴＵＶＷＸＹＺ"
    han = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    s = s.translate(str.maketrans(zen, han))

    return s

In [ ]:
def ensure_crs(gdf, epsg):
    if gdf.crs is None:
        raise ValueError("Input layer has no CRS; assign its source CRS before reprojecting")
    return gdf.to_crs(epsg=epsg)


def assign_points_to_regions(points, regions, id_col="region_id", name_col="region_nm"):
    """Assign each point once; use nearest polygon only for boundary/unmatched points."""
    left = points.copy().reset_index(drop=False).rename(columns={"index": "_point_id"})
    right_cols = list(dict.fromkeys([id_col, name_col, "geometry"]))
    right = regions[right_cols].copy()

    joined = gpd.sjoin(left, right, how="left", predicate="within")
    matched_ids = set(joined.loc[joined[id_col].notna(), "_point_id"])
    unmatched = left.loc[~left["_point_id"].isin(matched_ids)].copy()

    if not unmatched.empty:
        nearest = gpd.sjoin_nearest(
            unmatched, right, how="left", max_distance=1.0, distance_col="boundary_dist_m"
        )
        joined = pd.concat([joined[joined[id_col].notna()], nearest], ignore_index=True)

    # A point exactly on a shared boundary may tie. Assign it deterministically once.
    joined[id_col] = joined[id_col].astype("string")
    joined = joined.sort_values(["_point_id", id_col], na_position="last")
    joined = joined.drop_duplicates("_point_id", keep="first")

    missing = int(joined[id_col].isna().sum())
    print(f"assigned {len(joined) - missing:,}/{len(left):,} points; unmatched: {missing:,}")
    return gpd.GeoDataFrame(joined, geometry="geometry", crs=points.crs)

### 2. Administrative boundary and registered population
### 2. 행정구역 경계 및 등록인구

Japan's e-Stat small-area statistics ship as a single file: the
administrative boundary geometry and the population/household counts
already live together, at chome (town) level -- no separate join needed
just to get a mappable population table.

일본 e-Stat의 소지역 통계는 한 파일로 제공된다 -- 경계 geometry와
인구·세대수가 처음부터 町丁목(마치) 단위로 한 파일에 함께 들어 있어서,
지도로 그릴 인구 테이블을 만들기 위해 별도로 매칭·병합할 필요가 없다.

This single-file structure is simply how e-Stat happens to publish this
dataset, not a shortcut we're taking -- worth noticing, since real
projects constantly run into "the data source isn't shaped the way you
expected."

이 한 파일 구조는 e-Stat이 이 데이터를 발행하는 방식일 뿐, 우리가 지름길을
택한 게 아니다 -- 실무에서는 "데이터 출처가 예상과 다른 구조"인 경우를 계속
마주치므로 눈여겨볼 만하다.


In [ ]:
ADMIN_GPKG = os.path.join(DATA_DIR, "hiroshima_city_admin.gpkg")  # boundary + population, combined
                                                                    # 경계 + 인구, 한 파일에 결합됨

ADMIN_COL_ID = "KEY_CODE"    # chome code column / 町丁목 코드 컬럼명
ADMIN_COL_NM = "region_nm"   # chome name column (already renamed on fetch) / 町丁목 이름 컬럼명 (수집 시 이미 리네임됨)

In [ ]:
gdf_admin = gpd.read_file(ADMIN_GPKG)  # read the boundary+population layer / 경계+인구 레이어 읽기
gdf_admin = gdf_admin[[ADMIN_COL_ID, ADMIN_COL_NM, "ward_nm", "pop", "households", "geometry"]].copy()
                                        # keep only the columns we need / 필요한 컬럼만 유지
gdf_admin.rename(columns={ADMIN_COL_ID: "region_id", "pop": "population"}, inplace=True)
                                        # standardize column names / 컬럼 이름을 표준 이름으로 변경

# 15 of the 1,136 rows share a region_id with another row: e-Stat splits some
# small areas into non-contiguous statistical fragments that keep the same
# KEY_CODE but carry different population values and different geometries
# (e.g. 温品町 appears once with population 0, once with 317). Every merge
# below treats region_id as a unique key -- left un-deduplicated, a bus/rail
# count aggregated for one region_id would get copied onto BOTH fragments via
# a one-to-many merge, crediting a stop to a fragment it does not serve. A classic
# non-unique-join-key bug: silently duplicated rows, no error raised.
# Dissolve to one row per region_id before anything else
# touches this table: population/households sum, geometry unions.
# 1,136행 중 15개는 다른 행과 region_id를 공유한다: e-Stat이 일부 소지역을
# KEY_CODE는 같지만 인구값과 geometry가 다른, 서로 붙어있지 않은 통계
# 조각들로 나눠놓았기 때문이다(예: 温品町이 인구 0인 행과 317인 행 둘로 존재).
# 아래의 모든 병합은 region_id를 유일 키로 취급한다 -- 중복을
# 제거하지 않으면, 한 region_id로 집계된 버스/철도 개수가 일대다 병합을 통해
# 두 조각 모두에 복사되어, 실제로는 서비스하지 않는 조각에까지 정류장이
# 계상된다. 비유일 조인 키의 전형적인 실패 유형이다 -- 에러 없이 행이 조용히
# 중복된다. 이후 어떤 작업이 닿기 전에, region_id 하나당 한 행으로 dissolve한다:
# 인구·세대수는 합산, geometry는 union.
_dupe_ids = gdf_admin["region_id"][gdf_admin["region_id"].duplicated()].unique()
if len(_dupe_ids) > 0:
    print(f"{len(_dupe_ids)} region_id value(s) appear on multiple rows -- dissolving before any merge")
    print(f"여러 행에 걸친 region_id {len(_dupe_ids)}개 -- 병합 전에 dissolve")

gdf_admin = gdf_admin.dissolve(
    by="region_id",
    aggfunc={"region_nm": "first", "ward_nm": "first", "population": "sum", "households": "sum"},
).reset_index()

assert gdf_admin["region_id"].is_unique, "region_id must be unique before any downstream merge"

gdf_admin.head()


In [ ]:
gdf_admin.plot(figsize=(8, 8), edgecolor="black")
plt.title("Hiroshima City small-area boundaries")
plt.axis("off")
plt.show()

In [ ]:
# Normalize chome names with norm_nm (unifies whitespace/full-width digits so
# later joins match correctly)
# 지명을 norm_nm으로 정규화 (공백·전각 숫자를 통일해 이후 데이터 매칭 정확도 향상)
gdf_admin["region_nm"] = gdf_admin["region_nm"].map(norm_nm)

In [ ]:
# Reproject to a metric CRS (EPSG:6670, JGD2000 / Plane Rectangular CS III --
# Hiroshima Prefecture's zone) so area, distance, and buffer calculations are
# all done in metres, not degrees.
# 미터 단위 좌표계(EPSG:6670, JGD2000 평면직각좌표계 -- 히로시마현이 속한
# 3계)로 변환한다. 이후 면적·거리·버퍼 계산을 모두 같은 좌표계, 미터 단위로
# 처리하기 위함이다.
gdf_admin = ensure_crs(gdf_admin, 6670)

In [ ]:
gdf_admin["area_km2"] = gdf_admin.geometry.area / 1e6  # m^2 -> km^2 / m^2를 km^2로 변환
gdf_admin["pop_density"] = gdf_admin["population"] / gdf_admin["area_km2"]  # people per km^2 / km^2당 인구
gdf_admin[["region_nm", "ward_nm", "population", "area_km2", "pop_density"]].head()

In [ ]:
ADMIN_FILE = os.path.join(DATA_DIR, "hiroshima_admin_pop.csv")  # save the combined table / 결합 테이블 저장 경로
gdf_admin.drop(columns="geometry").to_csv(ADMIN_FILE, index=False, encoding="utf-8-sig")
                                        # drop geometry for the plain CSV export / CSV로는 geometry 제외하고 저장

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

gdf_admin.plot(
    column="pop_density",      # color by this column / 색칠 기준 컬럼 (인구밀도)
    cmap="OrRd",                # light -> dark palette / 색상 팔레트 (연한색 → 진한색)
    legend=True,
    edgecolor="black",
    linewidth=0.2,
    ax=ax,
)

plt.title("Hiroshima City population density by small area", fontsize=13)
plt.axis("off")
plt.show()

#### Coordinate Reference Systems (CRS)

- A reference system for expressing a location on Earth as flat coordinates
- Broadly split into geographic (lat/lon) and projected coordinate systems
- Spatial layers with different CRSs must be reprojected to a common one before analysis
- CRS choice materially affects the accuracy of distance, area, and buffer calculations
- The right CRS depends on the analysis goal -- there's no single "correct" one

#### 좌표계(CRS, Coordinate Reference System)

- 지구상의 위치를 평면 좌표로 표현하기 위한 기준 체계
- 위도·경도 좌표계와 투영 좌표계로 크게 구분됨
- 공간 데이터 간의 좌표계가 다르면 반드시 변환 후 분석해야 함
- 거리, 면적, 버퍼 계산의 정확도는 좌표계 선택에 크게 영향을 받음
- GIS 분석에서는 "분석 목적에 맞는 좌표계"를 선택하는 것이 중요

#### EPSG:4326 (WGS84, latitude/longitude)

- The global-standard geographic CRS, expressed in latitude and longitude
- Units are degrees, so it's unsuitable for distance/area calculations
- The default CRS for GPS, map APIs, and most raw location data
- Convenient for display, but imprecise for rigorous spatial analysis

#### EPSG:4326 (WGS84, 위경도 좌표계)

- 위도(latitude)와 경도(longitude)로 위치를 표현하는 전 지구 표준 좌표계
- 단위가 '도(degree)'이기 때문에 거리·면적 계산에 부적합
- GPS, 지도 API, 대부분의 원본 위치 데이터의 기본 좌표계
- 시각화·위치 표시에는 편리하지만 정밀 공간 분석에는 부정확

#### EPSG:6670 (JGD2000 / Plane Rectangular CS III)

- A projected CRS covering Yamaguchi, Shimane, and Hiroshima prefectures
- Units are metres, so distance/area/buffer calculations are accurate
- Suited to administrative-area calculations, accessibility analysis, coverage analysis
- Japan defines 19 such zones nationwide; each region uses its own zone number

#### EPSG:6670 (JGD2000 평면직각좌표계 3계)

- 야마구치·시마네·히로시마현을 포함하는 투영 좌표계
- 단위가 '미터(m)'이므로 거리·면적·버퍼 계산에 정확
- 행정구역 면적 계산, 접근성 분석, 커버리지 분석 등에 적합
- 일본 전역에 19개 계로 나뉘어 있으며, 지역마다 자신의 계 번호를 사용

#### Practical guidance

- Loading raw data: keep it in EPSG:4326 as-is
- Before spatial analysis: standardize to EPSG:6670 (or the projected CRS for your region)
- Area/distance/buffer calculations: always run them in a projected CRS
- EPSG.io (https://epsg.io)

#### 실무 권장 원칙

- 원본 데이터 불러오기: EPSG:4326 그대로 사용
- 공간 분석 시작 전: EPSG:6670(또는 해당 지역의 투영 좌표계)로 통일
- 면적·거리·버퍼 계산은 항상 투영 좌표계에서 수행
- EPSG.io (https://epsg.io)

### 3. Bus and rail/streetcar facility aggregation
### 3. 버스 및 철도/노면전차 시설 집계

Here we turn bus-stop and station locations into spatial objects, then spatial-join
them against the administrative boundary polygons to count facilities per area.

여기서는 버스정류장과 철도역 위치를 공간 객체로 변환한 뒤, 행정구역 경계
폴리곤과의 공간 조인을 통해 각 구역별 시설 개수를 집계한다.

KSJ's bus-stop and station layers already ship as GIS files with geometry
included, so there's no need to build Point geometry from raw X/Y columns
in a CSV/Excel file first. One less step, but the CRS conversion and
spatial-join logic taught here are the same either way.

国土数値情報(KSJ)의 버스정류장·역 레이어는 처음부터 geometry가 포함된
GIS 파일로 제공되므로, CSV/엑셀의 원본 X/Y 컬럼으로 Point geometry를
직접 만들 필요가 없다. 한 단계가 줄어들 뿐, 이후 배우는 좌표계 변환과
공간조인 로직은 동일하다.


In [ ]:
BUS_GPKG = os.path.join(DATA_DIR, "hiroshima_city_bus_stops.gpkg")  # bus stop locations / 버스정류장 위치 파일 경로

gdf_bus = gpd.read_file(BUS_GPKG)  # already has Point geometry -- no X/Y columns to build from
                                     # 이미 Point geometry가 있어 X/Y 컬럼으로 만들 필요 없음
gdf_bus = gdf_bus.to_crs(epsg=6670)  # reproject to the metric CRS / 미터 단위 좌표계로 변환
gdf_bus[["P11_001", "geometry"]].head()  # P11_001 = stop name / P11_001 = 정류장 이름

In [ ]:
bus_joined = assign_points_to_regions(gdf_bus, gdf_admin)
bus_cnt = (
    bus_joined.dropna(subset=["region_id"])
    .groupby(["region_id", "region_nm"], dropna=False)
    .size()
    .reset_index(name="bus_stop_cnt")
)

assert bus_cnt["bus_stop_cnt"].sum() <= len(gdf_bus)
print(f"bus stops counted: {int(bus_cnt['bus_stop_cnt'].sum()):,}/{len(gdf_bus):,}")
bus_cnt.head()

In [ ]:
gdf_bus_map = gdf_admin.merge(
    bus_cnt[["region_id", "bus_stop_cnt"]],
    on="region_id",
    how="left",
)
# join the count back onto the boundary layer, on chome code
# 경계 데이터에 버스정류장 집계 결과를 소지역 코드 기준으로 병합하여 지도용 데이터 생성

gdf_bus_map["bus_stop_cnt"] = gdf_bus_map["bus_stop_cnt"].fillna(0)
# small areas with zero stops become 0, not NaN, to avoid downstream errors
# 버스정류장이 없는 소지역은 NaN 대신 0으로 채워서 이후 시각화·계산 오류 방지

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

gdf_bus_map.plot(
    column="bus_stop_cnt",
    cmap="Blues",
    legend=True,
    edgecolor="black",
    linewidth=0.2,
    ax=ax,
)

plt.title("Bus stop count by small area, Hiroshima City", fontsize=13)
plt.axis("off")
plt.show()

In [ ]:
BUS_FILE = os.path.join(DATA_DIR, "bus_stop_cnt_by_admin.csv")  # save path / 집계 결과 저장 경로
bus_cnt.to_csv(BUS_FILE, index=False, encoding="utf-8-sig")

In [ ]:
STATION_GPKG = os.path.join(DATA_DIR, "hiroshima_city_stations.gpkg")  # rail/streetcar station locations
                                                                          # 철도/노면전차역 위치 파일 경로

gdf_station = gpd.read_file(STATION_GPKG)
gdf_station[["N02_003", "N02_004", "N02_005", "geometry"]].head()
# N02_003 = line name, N02_004 = operator, N02_005 = station name
# N02_003 = 노선명, N02_004 = 사업자명, N02_005 = 역명

In [ ]:
# N02 station geometry is a short platform LineString (e.g. Hiroden's
# dual-track streetcar stops), not a point. Take the centroid, in a metric
# CRS, for a clean marker -- the same fix used in hiroshima_slope_case.ipynb.
# N02 역 geometry는 점이 아니라 짧은 플랫폼 선분(広電 복선 노면전차 정류장 등)
# 이다. 미터 좌표계에서 centroid를 구해 깔끔한 마커로 쓴다 --
# hiroshima_slope_case.ipynb에서 쓴 것과 같은 처리다.
gdf_station = gdf_station.to_crs(epsg=6670)
gdf_station["geometry"] = gdf_station.geometry.centroid

In [ ]:
ax = gdf_admin.plot(edgecolor="black", facecolor="none", figsize=(8, 8))
# draw the small-area boundaries first / 행정구역 경계를 먼저 그림

gdf_station.plot(
    ax=ax,
    color="red",
    markersize=5,
)
# rail/streetcar stations as red points / 철도/노면전차역 포인트를 빨간 점으로 표시

plt.title("Hiroshima City boundaries and rail/streetcar stations")
plt.axis("off")
plt.show()

In [ ]:
station_joined = assign_points_to_regions(gdf_station, gdf_admin)
station_cnt = (
    station_joined.dropna(subset=["region_id"])
    .groupby(["region_id", "region_nm"], dropna=False)
    .size()
    .reset_index(name="station_cnt")
)

assert station_cnt["station_cnt"].sum() <= len(gdf_station)
print(f"stations counted: {int(station_cnt['station_cnt'].sum()):,}/{len(gdf_station):,}")

STATION_FILE = DATA_DIR / "station_cnt_by_admin.csv"
station_cnt.to_csv(STATION_FILE, index=False, encoding="utf-8-sig")

### 4. Merging processed data and building derived indicators
### 4. 가공 데이터 병합 및 파생지표

Here we merge the population, bus, and rail/streetcar tables we've built so
far into one combined table, keyed on chome code (region_id).

지금까지 개별적으로 가공한 인구, 버스, 철도/노면전차 데이터를 소지역 코드
(region_id) 기준으로 하나의 통합 테이블로 병합한다.

To correct for population-size differences, we build normalized indicators:
bus stops per 10,000 residents, and rail/streetcar stations per 10,000
residents. These go beyond a raw headcount, letting us compare accessibility
gaps between areas on equal footing -- the core metric for identifying
transit-underserved areas later.

인구 규모 차이를 보정하기 위해 인구 1만 명당 버스정류장 수, 인구 1만 명당
철도/노면전차역 수 같은 표준화 지표를 만든다. 단순 개수 비교를 넘어 지역
간 접근성 격차를 동일한 기준으로 비교할 수 있게 되며, 이는 이후 교통
취약지역을 선별하는 핵심 지표가 된다.

In [ ]:
pop_n = gdf_admin[["region_id", "population", "pop_density"]].copy()
bus_n = bus_cnt[["region_id", "bus_stop_cnt"]].copy()
sub_n = station_cnt[["region_id", "station_cnt"]].copy()

df = (
    pop_n
    .merge(bus_n, on="region_id", how="left")
    .merge(sub_n, on="region_id", how="left")
)

df["bus_stop_cnt"] = df["bus_stop_cnt"].fillna(0)
df["station_cnt"] = df["station_cnt"].fillna(0)
df.head()

In [ ]:
gdf_map = gdf_admin.merge(
    df[["region_id", "bus_stop_cnt", "station_cnt"]],
    on="region_id",
    how="left",
)

# Some small areas have zero registered residents (parks, industrial land, etc.).
# This is a real characteristic of Japan's
# fine chome-level subdivision.
# "Stops per 10,000 residents" is undefined with zero residents, so those
# areas get NaN here rather than a division-by-zero inf/nan silently leaking
# into the analysis.
# 히로시마 1,136개 소지역 중 93곳은 등록 인구가 0이다(공원, 공업지역, 공항
# 부지 등) -- 일본의 촘촘한 町丁목 단위 세분화가 만든 실제 특성이다. 인구가 0이면 "인구 1만명당"
# 지표 자체가 정의되지 않으므로, 0으로 나눠 나오는 inf/nan이 조용히 분석에
# 섞이지 않도록 이 지역들은 NaN으로 명시적으로 남긴다.
has_residents = gdf_map["population"] > 0
gdf_map["bus_per_10k"] = np.where(
    has_residents, gdf_map["bus_stop_cnt"] / (gdf_map["population"] / 10000), np.nan
)
gdf_map["station_per_10k"] = np.where(
    has_residents, gdf_map["station_cnt"] / (gdf_map["population"] / 10000), np.nan
)
print(f"{(~has_residents).sum()} small areas have zero population -- excluded from per-capita indicators")
print(f"인구 0인 소지역 {(~has_residents).sum()}곳 -- 1인당 지표에서 제외")
gdf_map.head()

In [ ]:
# 40% of populated small areas have exactly zero bus stops within their own
# polygon -- Hiroshima's chome subdivision is fine-grained enough that most
# residents rely on a stop in a *neighbouring* block, not their own. A plain
# 5-quantile qcut can't split a distribution where the bottom 40% of values
# are identical (it errors on duplicate bin edges). So "no stop in this
# block" gets its own explicit class, and only the small areas that do have
# a stop get quantile-split into low/medium/high.
# 인구가 있는 소지역의 40%는 자기 폴리곤 안에 버스정류장이 아예 없다 --
# 히로시마의 町丁목 세분화가 촘촘해서 대부분 주민이 '이웃' 블록의 정류장을
# 이용하기 때문이다. 하위 40%가 전부 같은 값이면 일반적인 5분위 qcut은
# 구간 경계 중복으로 에러가 난다. 그래서 "이 블록엔 정류장 없음"을 별도
# 범주로 명시하고, 정류장이 있는 소지역만 low/medium/high로 분위수 분할한다.
has_stop = gdf_map["bus_per_10k"] > 0
gdf_map["bus_class"] = pd.Series(pd.NA, index=gdf_map.index, dtype="object")
gdf_map.loc[gdf_map["bus_per_10k"] == 0, "bus_class"] = "no stop in block"
gdf_map.loc[has_stop, "bus_class"] = pd.qcut(
    gdf_map.loc[has_stop, "bus_per_10k"],
    q=3,
    labels=["low", "medium", "high"],
)

# Without an explicit order, geopandas colors categories alphabetically --
# "high" would render lightest and "no stop in block" darkest, exactly
# backwards from what the Blues gradient should mean. Force the real order.
# 순서를 명시하지 않으면 geopandas가 알파벳순으로 색을 매기는데, 그러면
# "high"가 가장 연하고 "no stop in block"이 가장 진하게 나와 Blues 그라데이션이
# 뜻하는 것과 정반대가 된다. 실제 순서를 강제로 지정한다.
gdf_map["bus_class"] = pd.Categorical(
    gdf_map["bus_class"],
    categories=["no stop in block", "low", "medium", "high"],
    ordered=True,
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

gdf_map.plot(
    column="bus_class",
    cmap="Blues",
    edgecolor="black",
    linewidth=0.2,
    legend=True,
    ax=ax,
    legend_kwds={
        "title": "Bus accessibility\n(per 10,000 residents)",
        "loc": "center left",
        "bbox_to_anchor": (1.02, 0.5),  # place legend outside the map, to the right
    },
)

plt.title("Bus accessibility per 10,000 residents, Hiroshima City", fontsize=13)
plt.axis("off")
plt.tight_layout()
plt.show()

Notice the mountainous fringe wards (top and left) read as "high" -- darkest
blue -- while the dense urban core along the river delta looks patchier.
That's not because the hills are better served: a per-capita ratio like
"stops per 10,000 residents" gets inflated wherever the *denominator* is
tiny. A hillside area with 2 residents and 1 through-route stop scores
5,000 per 10k; a dense downtown block with 2,000 residents and 20 stops
scores 100. Both numbers are correct, but only one describes what a rider
actually experiences. This is the same shape of problem area-weighted per-capita indicators
run into everywhere: a number that is technically correct can still
describe the wrong thing.

산간 외곽 구(위·왼쪽)가 가장 진한 파랑, 즉 "high"로 나오는데, 언덕 지역의
서비스가 실제로 더 좋아서가 아니다. "인구 1만명당" 같은 1인당 비율은
분모(인구)가 작을수록 부풀려진다. 주민 2명에 통과노선 정류장 1개가 있는
산간 지역은 1만명당 5,000이 나오지만, 주민 2,000명에 정류장 20개가 있는
번화가는 1만명당 100이 나온다. 둘 다 계산은 맞지만, 실제 이용자 경험을
설명하는 건 후자뿐이다. 1인당 지표가 흔히 겪는 것과 같은 종류의 문제다 --
숫자 자체는 틀리지 않았는데 엉뚱한 것을 설명하고 있다.


In [ ]:
OUT_GPKG = os.path.join(OUT_DIR, "hiroshima_eda_map.gpkg")

gdf_map.to_file(OUT_GPKG, layer="eda_map", driver="GPKG")
# save boundary + population + bus + rail + derived indicators together
# 경계 + 인구 + 버스 + 철도 + 파생지표를 포함한 공간 데이터 저장
# can be reopened directly in QGIS or GeoPandas
# QGIS, GeoPandas에서 바로 다시 불러와 지도 시각화 가능

#### Shapefile (.shp) vs. GeoPackage (.gpkg)

Shapefile and GeoPackage are both common spatial-data formats, but differ
structurally. A Shapefile layer is spread across several files (.shp, .shx,
.dbf, .prj, ...), which is awkward to manage, and it has real constraints --
a 10-character column-name limit and encoding issues with non-Latin text.
Still, as an old standard, it's widely supported across GIS software.

GeoPackage is a modern OGC-standard format that stores everything in a
single file. It supports multiple layers at once, has no column-length
limit, and defaults to UTF-8 -- handling Japanese and Korean text natively.
It performs well on large data and is much easier to manage.

Shapefile과 GeoPackage는 공간 데이터를 저장하는 대표적인 형식이지만 구조와
기능에서 큰 차이를 가진다.

Shapefile은 하나의 레이어가 여러 개의 파일(.shp, .shx, .dbf, .prj 등)로
구성되어 관리가 불편하고, 컬럼 이름 10자 제한과 비라틴 문자 인코딩 문제 등
구조적 제약이 많다. 다만 오래된 표준 형식으로 대부분의 GIS 소프트웨어에서
널리 지원된다.

GeoPackage는 하나의 파일 안에 모든 정보를 저장하는 최신 OGC 표준 형식으로,
여러 레이어를 동시에 저장할 수 있고 컬럼 길이 제한이 없으며 UTF-8 인코딩을
기본 지원해 일본어·한국어 텍스트를 그대로 다룰 수 있다. 대용량 데이터
처리 성능도 우수하고 관리가 매우 편리하다.

**Recommendation**: Shapefile for legacy-system compatibility or simple
layer hand-off; GeoPackage for teaching, research, real-world work, and
anything involving non-Latin text or combined analysis results.

**활용 권장**: Shapefile은 구형 시스템 호환이 필요하거나 단순 레이어
전달용, GeoPackage는 강의·연구·실무 전반, 비라틴 문자 데이터 포함, 복합
분석 결과 저장용.

### 5. Exploratory Data Analysis (EDA) / 5. 탐색적 데이터 분석(EDA)

#### "Do areas with low bus accessibility also have low rail/streetcar accessibility?"
#### "버스 접근성이 낮은 지역은 철도/노면전차 접근성도 낮은가?"


In [ ]:
# ~40% of populated small areas tie at bus_per_10k == 0 (see the note above --
# most residents rely on a neighbouring block's stop). A plain ascending sort
# would pick 10 of those ties arbitrarily, so break ties by population
# descending: among equally zero-served areas, surface the ones affecting
# the most residents first -- the more useful "worst 10" for a real audience.
# 인구가 있는 소지역의 약 40%가 bus_per_10k == 0으로 동률이다(위 설명대로
# 대부분 이웃 블록의 정류장을 이용하기 때문). 그냥 오름차순 정렬만 하면 이
# 동률 중 임의의 10곳이 뽑힌다 -- 인구 내림차순으로 동률을 깨서, 똑같이
# 서비스가 없는 지역 중에서도 더 많은 주민에게 영향을 주는 곳을 먼저 보여준다.
target_top10 = gdf_map.sort_values(
    ["bus_per_10k", "station_per_10k", "population"],
    ascending=[True, True, False],
).head(10)

# A handful of very-low-population small areas have extreme per-capita
# ratios (e.g. 1 stop / 1 resident = 10,000 per 10k) that stretch the axes
# and hide the real pattern -- clip the view, not the underlying data.
# 인구가 극히 적은 소지역 몇 곳은 1인당 비율이 극단적으로 커져(예: 주민 1명에
# 정류장 1개 = 인구 1만명당 10,000) 축을 왜곡시킨다 -- 원본 데이터는 그대로
# 두고 보이는 범위만 clip한다.
plt.figure(figsize=(8, 7))
plt.scatter(
    gdf_map["bus_per_10k"],
    gdf_map["station_per_10k"],
    alpha=0.5,  # point transparency, to reduce overplotting / 점 투명도 조절 (겹침 완화)
)

# highlight the worst-10 in red
# 접근성 최하위 10곳만 빨간색으로 강조 표시
plt.scatter(
    target_top10["bus_per_10k"],
    target_top10["station_per_10k"],
    color="red",
    s=70,
    label="Bottom 10",
)

# label the worst-10 with their names
# 최하위 10곳에 지명 라벨 표시
for _, row in target_top10.iterrows():
    plt.annotate(
        row["region_nm"],
        (row["bus_per_10k"], row["station_per_10k"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9,
        color="red",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8),
    )

plt.xlim(-50, 1000)   # clip the view past the small-denominator outliers / 소분모 이상치 너머는 보이는 범위에서 제외
plt.ylim(-50, 1000)
plt.xlabel("Bus stops per 10,000 residents")
plt.ylabel("Rail/streetcar stations per 10,000 residents")
plt.title("Bottom-10 small areas by transit accessibility")
plt.legend()
plt.show()

In [ ]:
target_top10[[
    "region_nm",        # small area name / 소지역 이름
    "ward_nm",           # ward / 소속 구
    "population",        # total population / 총 인구
    "bus_stop_cnt",       # bus stop count / 버스정류장 개수
    "bus_per_10k",        # bus stops per 10k / 인구 1만 명당 버스정류장 수
    "station_cnt",        # station count / 역 개수
    "station_per_10k",    # stations per 10k / 인구 1만 명당 역 수
]]

#### "Where population density is high, is bus access sufficient?"
#### "인구 밀도가 높은 지역은 버스 접근성이 충분한가?"


In [ ]:
plt.figure(figsize=(8, 7))  # figure size / 그래프 크기 설정

# scatter every small area: population density vs. bus accessibility
# 전체 소지역 산점도 그리기 (인구밀도 vs 버스 접근성)
plt.scatter(
    gdf_map["pop_density"],   # x: population density / x축: 인구밀도
    gdf_map["bus_per_10k"],   # y: bus stops per 10k residents / y축: 인구 1만명당 버스 정류장 수
    alpha=0.5,
)

# threshold definitions
# 기준값 설정
high_density = gdf_map["pop_density"].quantile(0.8)   # top 20% population density / 인구밀도 상위 20% 기준값
low_bus      = gdf_map["bus_per_10k"].quantile(0.1)    # bottom 10% bus accessibility / 버스 접근성 하위 10% 기준값

# candidate underserved areas: high density AND low bus access
# 취약 후보 지역 선택: 인구밀도는 높고 + 버스 접근성은 낮은 소지역
target2 = gdf_map[
    (gdf_map["pop_density"] >= high_density) &
    (gdf_map["bus_per_10k"] <= low_bus)
]

# highlight the candidates in red
# 취약 지역만 빨간색으로 강조 표시
plt.scatter(
    target2["pop_density"],
    target2["bus_per_10k"],
    color="red",
    s=60,
    label="High density, low bus access",
)

print(f"{len(target2)} small areas match both thresholds -- too many to label individually")
print(f"두 기준을 모두 만족하는 소지역 {len(target2)}곳 -- 전부 라벨링하기엔 너무 많음")

# label only the 5 most extreme (highest density) candidates, or the plot
# turns into unreadable overlapping text
# 가장 극단적인(밀도가 가장 높은) 5곳만 라벨링한다 -- 전부 표시하면 텍스트가
# 겹쳐 읽을 수 없다
for _, row in target2.nlargest(5, "pop_density").iterrows():
    plt.annotate(
        row["region_nm"],
        (row["pop_density"], row["bus_per_10k"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9,
        color="red",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8),
    )

plt.ylim(-50, 1000)  # clip past the small-denominator outliers, same as the previous plot
                       # 이전 그래프와 마찬가지로 소분모 이상치 너머는 보이는 범위에서 제외
plt.xlabel("Population density (people/km^2)")
plt.ylabel("Bus stops per 10,000 residents")
plt.title("High-density, low-bus-access small areas")
plt.legend()
plt.show()

### Lecture 1 summary / 1강 요약

**Key takeaways / 핵심 요약**
- Turned raw files (boundaries + population + stop locations) into one analysis-ready GeoDataFrame via CRS unification, spatial joins, and per-area aggregation.
- Built population-normalized indicators (`bus_per_10k`, `station_per_10k`) so areas of very different size/population can be compared fairly.
- Handled two real messy-data edge cases head-on instead of hiding them: zero-population areas (excluded from per-capita indicators, not divided by zero) and zero-stop areas (their own explicit category, not lumped into the lowest quantile bin).

**핵심 요약**
- 원본 파일(경계, 인구, 정류장 위치)을 좌표계 통일, 공간조인, 구역별 집계를 거쳐 하나의 분석용 GeoDataFrame으로 만들었다.
- 인구 대비 지표(`bus_per_10k`, `station_per_10k`)를 만들어서 규모가 다른 지역도 공정하게 비교할 수 있게 했다.
- 실제 데이터의 지저분한 두 경우를 숨기지 않고 정면으로 다루었다: 인구가 0인 지역(0으로 나누지 않고 1인당 지표에서 제외), 정류장이 0개인 지역(최하위 구간에 섞지 않고 별도 항목으로 표시).

**Real-world tip / 실무 팁**
The `region_id` dissolve near the top of this lecture isn't decoration -- it fixes a real bug this exact notebook shipped with. 15 of Hiroshima's 1,136 small areas turned out to share a `region_id` across multiple rows (split statistical fragments), and every merge below silently treated it as unique. The result: bus-stop counts got copied onto the wrong fragment. Whenever you merge or groupby-then-merge-back on a key, check `df[key].is_unique` first -- it's a one-line check that would have caught this immediately.

**실무 팁**
이 강의 앞부분의 `region_id` dissolve 작업은 장식이 아니다. 이 노트북이 실제로 가지고 있던 버그를 고친 것이다. 히로시마의 1,136개 소지역 중 15개가 여러 행에 걸쳐 같은 `region_id`를 공유하고 있었다. e-Stat이 통계상 조각낸 것인데, 아래의 모든 병합 코드는 이 값을 유일한 값처럼 취급하고 있었다. 그 결과 버스정류장 개수가 엉뚱한 조각에 복사되었다. 키를 기준으로 merge하거나 groupby 후 다시 merge-back할 때는 항상 먼저 `df[key].is_unique`를 확인하자.

**Try it yourself / 직접 해보기**
Insert a new cell below and make a copy of the per-capita split with a different number of quantile bins. Does the "no stop in block" category still dominate the map?

아래에 새 셀을 추가해서, 분위 개수를 다르게 바꾼 사본을 만들어보자. "정류장 없음" 항목이 여전히 지도를 지배하는가?

<details>
<summary>Show one way to do it / 정답 예시 보기</summary>

```python
my_bus_class = pd.qcut(
    gdf_map.loc[gdf_map["bus_per_10k"] > 0, "bus_per_10k"],
    q=5,  # try 5 instead of 3
    labels=["very low", "low", "medium", "high", "very high"],
)
my_bus_class.value_counts()
```

</details>

---
### Break (10 min) / 휴식 (10분)

Lecture 1 is done. Take a short break -- stretch, grab water, ask a
neighbour a question. We'll start Lecture 2 in 10 minutes.

Finished early? Try changing the `q=3` split in the `bus_class` cell
above to `q=5`, or pick a different ward and see how the population-
density map changes.

1강이 종료되었다. 잠깐 쉬자 -- 스트레칭, 물 한잔, 옆 사람과 질문
나누기. 10분 뒤 2강을 시작한다.

먼저 끝났다면? 위 `bus_class` 셀의 `q=3` 분할을 `q=5`로 바꿔보거나,
다른 구를 골라 인구밀도 지도가 어떻게 바뀌는지 직접 시도해보자.
---

### [Lecture 2] Separating "reached" from "not reached"
### [2강] 닿는 곳과 닿지 않는 곳을 공간으로 분리하다
#### Buffer-based coverage / non-coverage analysis (GeoPandas)
#### 버퍼 기반 커버 / 비커버 분석 (GeoPandas)

Lecture 1 aggregated bus and rail/streetcar stop *counts* by small area, and
built population-normalized indicators (bus_per_10k, station_per_10k) from
them. Lecture 2 asks the same underlying question a different way: not "how
many stops does this area have," but "how much of this area's actual land
is within walking distance of a stop."

1강은 소지역 단위로 버스·철도/노면전차 정류장 *개수*를 집계하고, 인구
대비 접근성 지표(bus_per_10k, station_per_10k)를 만들었다. 2강은 같은
질문을 다른 방식으로 다시 묻는다: "이 지역에 정류장이 몇 개 있는가"가
아니라 "이 지역의 실제 면적 중 얼마나 많은 부분이 정류장 도보권 안에
있는가."

Space within a set distance (buffer) of a bus stop or station is defined as
*covered*; everything else is *uncovered*. We build the bus buffer and the
rail/streetcar buffer separately, then merge them into one combined coverage
area (bus UNION rail). Subtracting that combined coverage from each small
area's polygon (a *difference* operation) produces the uncovered pockets
inside it. Finally we combine population density, uncovered ratio, and stop
count into one vulnerability score, and pick the top-ranked small areas as
candidates for improvement.

버스정류장 또는 철도/노면전차역으로부터 일정 거리(버퍼) 안에 포함되는
공간은 커버 영역(covered), 그 밖은 비커버 영역(uncovered)으로 정의한다.
버스 버퍼와 철도/노면전차 버퍼를 각각 만든 뒤 하나의 통합 커버리지(버스
∪ 철도)로 합친다. 이 통합 커버리지를 각 소지역 폴리곤에서 빼는(차집합)
연산으로 소지역 내부의 비커버 공간을 뽑아낸다. 마지막으로 인구밀도,
비커버 비율, 정류장 수를 종합한 취약 점수를 계산해 개선 우선순위가
높은 소지역을 선정한다.

The two demo small areas used for the rest of this tutorial aren't
hand-picked, either -- they fall out of whatever this lecture's ranking
happens to produce, and rerunning this lecture on different data would
pick different areas automatically.

이 튜토리얼 뒤에서 쓰는 두 데모 소지역도 손으로 고른 게 아니라, 이
강의가 계산하는 랭킹이 무엇을 내놓든 그대로 나온 결과다 -- 다른
데이터로 이 강의를 다시 돌리면 자동으로 다른 지역이 선정된다.


#### Walking-access radius standard
#### 보행 접근권 설정 기준

We use 300m for bus stops and 500m for rail/streetcar stations as the
effective walking-access radius, matching the 3-5 minute (bus) / 7-10
minute (rail) walk-time guidance common in transit-oriented-development
planning.

버스 정류장은 300m, 철도/노면전차역은 500m를 유효 접근 반경으로 쓴다.
이는 도보 3~5분(버스), 도보 7~10분(철도)이라는 TOD(대중교통 중심 개발)
가이드라인의 통상적인 기준을 반영한 것이다.

In [ ]:
# copy just the bus stop points, for buffering
# 버스정류장 포인트만 따로 복사해서 버퍼 분석용 GeoDataFrame을 만든다
gdf_bus_buf = gdf_bus[["geometry"]].copy()

# 300m circular buffer around each bus stop
# 각 버스정류장을 중심으로 반경 300m 버퍼(원형 폴리곤)를 생성한다
gdf_bus_buf["geometry"] = gdf_bus_buf.geometry.buffer(300)
gdf_bus_buf.head()

In [ ]:
# copy just the station points, for buffering
# 역 포인트만 복사해서 버퍼 분석용 GeoDataFrame을 만든다
gdf_station_buf = gdf_station[["geometry"]].copy()

# 500m circular buffer around each station
# 각 역을 중심으로 반경 500m 버퍼 폴리곤을 생성한다
gdf_station_buf["geometry"] = gdf_station_buf.geometry.buffer(500)
gdf_station_buf.head()

In [ ]:
# draw the small-area boundaries as background
# 소지역 경계를 먼저 배경으로 그린다
ax = gdf_admin.plot(edgecolor="black", facecolor="none", figsize=(8, 8))

# only a sample of 200 bus buffers, in blue -- drawing all of them would be
# too dense to read
# 버스 버퍼 중 200개만 파란색으로 표시 (전부 그리면 너무 겹쳐서 알아보기 어려움)
gdf_bus_buf.sample(min(200, len(gdf_bus_buf)), random_state=7).plot(
    ax=ax, facecolor="none", edgecolor="blue", linewidth=0.3,
)

# station buffers, in red (there are few enough to show them all)
# 역 버퍼는 빨간색으로 표시 (개수가 적어 전부 그려도 무방하다)
gdf_station_buf.plot(
    ax=ax, facecolor="none", edgecolor="red", linewidth=0.3,
)

plt.axis("off")
plt.show()

#### Basic spatial operations (Union / Intersection / Difference)
#### 공간 연산 기본 개념 (Union · Intersection · Difference)

- **Union**: `A.union(B)` -- everything covered by A or B combined (A ∪ B)
- **Intersection**: `A.intersection(B)` -- only the part where A and B overlap (A ∩ B)
- **Difference**: `A.difference(B)` -- what's left of A after removing B (A − B)

- **Union (합집합)**: `A.union(B)` -- A와 B를 모두 합친 전체 영역 (A ∪ B)
- **Intersection (교집합)**: `A.intersection(B)` -- A와 B가 겹치는 공통 부분 (A ∩ B)
- **Difference (차집합)**: `A.difference(B)` -- A에서 B를 제거하고 남은 부분 (A − B)

These three operations underlie accessibility analysis, coverage
calculation, and extracting uncovered ("blind spot") areas.

이 세 연산은 접근성 분석, 커버리지 계산, 비커버(사각지대) 영역 추출 등에
공통으로 쓰인다.

#### Building the combined coverage / 통합 커버리지 만들기

Bus buffers and station buffers were built separately in the cells above
(bus: 300m, rail/streetcar: 500m). The next few cells merge each set into
one polygon (`union_all()`), then combine bus-coverage and
station-coverage into a single "reachable by either" polygon --
`bus_union_geom.union(station_union_geom)`. Everything downstream in this
lecture is measured against that one combined shape.

버스 버퍼와 역 버퍼는 위에서 각각 만들었다(버스 300m, 철도/노면전차
500m). 아래 몇 셀은 각 버퍼 집합을 하나의 폴리곤으로 합치고
(`union_all()`), 다시 버스 커버리지와 철도 커버리지를 "둘 중 하나라도
닿는" 하나의 폴리곤으로 합친다 -- `bus_union_geom.union(station_union_geom)`.
이 강의의 나머지 계산은 전부 이 통합 도형 하나를 기준으로 한다.


In [ ]:
# merge every bus buffer into one (multi)polygon
# 버스 버퍼 여러 개를 하나의(멀티) 폴리곤으로 합친다
bus_union_geom = gdf_bus_buf.geometry.union_all()
bus_union_geom

In [ ]:
# same for the station buffers
# 역 버퍼도 동일하게 하나의 커버리지 폴리곤으로 합친다
station_union_geom = gdf_station_buf.geometry.union_all()
station_union_geom

In [ ]:
# final combined coverage: bus coverage UNION station coverage --
# "everywhere reachable by either a bus stop or a station"
# 최종 통합 커버리지: 버스 커버리지 ∪ 철도 커버리지 --
# "버스나 철도 중 하나라도 닿는 전체 영역"
coverage_union_geom = bus_union_geom.union(station_union_geom)
coverage_union_geom

In [ ]:
# wrap the combined coverage geometry in a GeoDataFrame so it plays nicely
# with plot / overlay / to_file
# 통합 커버리지 geometry를 GeoDataFrame으로 감싼다 (plot/overlay/to_file에서 쓰기 위함)
gdf_coverage_union = gpd.GeoDataFrame(
    {
        "name": ["bus_or_station_coverage_union"],
        "geometry": [coverage_union_geom],
    },
    crs=6670,
)
gdf_coverage_union

In [ ]:
# copy the lecture-1 result to build the vulnerability analysis on top of it
# 1강 결과를 복사해 취약지역 분석용 데이터프레임을 만든다
gdf_vuln = gdf_map.copy()

# each small area's total area, in m^2
# 각 소지역의 전체 면적(m^2) 계산
gdf_vuln["admin_area_m2"] = gdf_vuln.geometry.area
gdf_vuln.sort_values("admin_area_m2", ascending=False).head()

#### Extracting the uncovered pockets / 비커버 영역 뽑아내기

With the combined coverage built, each small area's uncovered space is
just what's left after subtracting that coverage from its polygon --
`.difference(coverage_union_geom)`. This is the *difference* operation
from the Union/Intersection/Difference primer above, applied for real:
the result is exact geometry, not a yes/no flag, so the next cells can
measure *how much* of each area is uncovered, not just *whether* any of
it is.

통합 커버리지를 만들었으니, 각 소지역의 비커버 공간은 그 폴리곤에서
커버리지를 뺀 나머지다 -- `.difference(coverage_union_geom)`. 앞서 배운
Union/Intersection/Difference 중 *차집합*을 실제로 적용하는 부분이다.
결과는 예/아니오 판정이 아니라 정확한 지오메트리이므로, 아래 셀들은
"비커버 영역이 있는가"가 아니라 "얼마나 있는가"를 잴 수 있다.


In [ ]:
# subtract the combined coverage from each small-area polygon -> what's left
# is the uncovered ("blind spot") space inside it
# 소지역 폴리곤에서 통합 커버리지를 빼서 비커버(닿지 않는) 영역을 만든다
gdf_vuln["uncovered_geom"] = gdf_vuln.geometry.difference(coverage_union_geom)
gdf_vuln.head()

In [ ]:
# a dedicated GeoDataFrame with the uncovered geometry as its geometry column
# 비커버 영역을 geometry로 쓰는 전용 GeoDataFrame 생성
gdf_uncovered = gpd.GeoDataFrame(
    gdf_vuln.drop(columns=["geometry"]).rename(columns={"uncovered_geom": "geometry"}),
    crs=gdf_vuln.crs,
)

fig, ax = plt.subplots(figsize=(8, 8))
gdf_uncovered.plot(ax=ax, color="red", alpha=0.25, edgecolor="none")
plt.title("Uncovered areas -- transit blind spots")
plt.axis("off")
plt.show()

# Persist the CITY-WIDE uncovered geometry. Lecture 4 must not reuse only the
# two demo polygons, otherwise surrounding areas are incorrectly treated as covered.
OUT_GPKG_UNCOVERED_CITY = OUT_DIR / "hiroshima_uncovered_city.gpkg"
gdf_uncovered.to_file(
    OUT_GPKG_UNCOVERED_CITY, layer="uncovered_city", driver="GPKG"
)
print("saved city-wide uncovered layer:", OUT_GPKG_UNCOVERED_CITY)

In [ ]:
# area of the uncovered portion, in m^2
# 비커버 영역의 면적(m^2) 계산
gdf_vuln["uncovered_area_m2"] = gdf_vuln["uncovered_geom"].area
gdf_vuln[["region_nm", "uncovered_area_m2"]].sort_values("uncovered_area_m2", ascending=False).head()

In [ ]:
# uncovered area as a fraction of the small area's total area
# np.where(condition, value_if_true, value_if_false)
# 전체 면적 대비 비커버 면적 비율 계산
# np.where(조건, 참일 때 값, 거짓일 때 값)
gdf_vuln["uncovered_ratio"] = np.where(
    gdf_vuln["admin_area_m2"] > 0,
    gdf_vuln["uncovered_area_m2"] / gdf_vuln["admin_area_m2"],
    0.0,
)
gdf_vuln[["region_nm", "uncovered_ratio"]].sort_values("uncovered_ratio", ascending=False).head()

In [ ]:
# fill any missing values with 0 and clip to the valid [0, 1] range
# 결측값(NaN)을 0으로 채우고, 값이 0~1 범위를 벗어나지 않도록 제한
gdf_vuln["uncovered_ratio"] = gdf_vuln["uncovered_ratio"].fillna(0.0).clip(0, 1)

#### Building the vulnerability score / 취약도 점수 만들기

Three signals feed the final score, each min-max scaled to a common 0-1
range so they can be added together on equal footing:

- **Population density** (`pop_density`) -- more residents in the same
  space, higher priority.
- **Uncovered ratio** (`uncovered_ratio`) -- more of the area's own land
  outside the combined coverage, more vulnerable.
- **Stop count** (`bus_stop_cnt + station_cnt`) -- inverted (`1 - x`)
  before scaling, since *fewer* stops should mean *more* vulnerable, not
  less.

The three are combined with equal weights (0.33/0.33/0.33) below --
`pop_norm*0.33 + unc_norm*0.33 + bus_station_inv_norm*0.33`. Equal
weighting is a starting assumption, not a fact about which factor matters
most; the lecture summary's "Try it yourself" exercise has you rerun this
exact cell with different weights and see how much the ranking moves.

세 가지 신호가 최종 점수를 만든다. 각각 0~1로 min-max 정규화한 뒤
동일한 기준으로 더한다:

- **인구밀도**(`pop_density`) -- 같은 면적에 거주자가 많을수록 우선순위가 높다.
- **비커버 비율**(`uncovered_ratio`) -- 통합 커버리지 밖에 있는 면적이
  넓을수록 취약하다.
- **정류장 수**(`bus_stop_cnt + station_cnt`) -- 정규화 전에 역전(`1 - x`)
  시킨다. 정류장이 *적을수록* 더 취약해야 하기 때문이다.

아래에서 세 지표를 동일 가중치(0.33/0.33/0.33)로 합친다 --
`pop_norm*0.33 + unc_norm*0.33 + bus_station_inv_norm*0.33`. 동일 가중치는
어느 요인이 가장 중요한지에 대한 사실이 아니라 하나의 출발 가정일
뿐이다 -- 강의 요약의 "직접 해보기" 실습에서 이 셀을 다른 가중치로
다시 돌려 순위가 얼마나 바뀌는지 직접 확인해본다.


In [ ]:
# min-max scaling to a common 0-1 range
# 0~1 범위로 정규화하는 min-max 스케일링 함수
def _minmax(s):
    s = s.astype(float)
    if s.max() == s.min():          # every value identical -> return all zeros
                                      # 모든 값이 동일하면 전부 0으로 반환
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

In [ ]:
# normalize population density and uncovered ratio so indicators on different
# scales can be compared directly
# 인구밀도와 비커버 비율을 0~1로 정규화해 서로 다른 단위의 지표를 비교 가능하게 만든다
gdf_vuln["pop_norm"] = _minmax(gdf_vuln["pop_density"])   # higher density = more vulnerable / 밀도가 높을수록 취약도 증가
gdf_vuln["unc_norm"] = _minmax(gdf_vuln["uncovered_ratio"])  # higher uncovered ratio = more vulnerable / 비커버 비율이 높을수록 취약도 증가

In [ ]:
# fewer bus stops + stations should mean MORE vulnerable, so invert the
# normalized value (1 - x)
# 정류장 수가 적을수록 더 취약해야 하므로, 정규화한 값을 1에서 빼서 역전시킨다
gdf_vuln["bus_station_inv_norm"] = 1 - _minmax(gdf_vuln["bus_stop_cnt"] + gdf_vuln["station_cnt"])

In [ ]:
# weights (how much each indicator contributes to the vulnerability score)
# - population density: more residents -> higher improvement priority -> 0.33
# - uncovered ratio: more uncovered land -> more vulnerable -> 0.33
# - bus + station count: fewer stops -> more vulnerable -> 0.33
# 가중치 설정 (각 지표가 취약도에 기여하는 비중)
# - 인구밀도: 거주자가 많을수록 개선 우선순위 높음 -> 0.33
# - 비커버 비율: 비커버 면적이 클수록 취약 -> 0.33
# - 버스+역 수: 정류장이 적을수록 취약 -> 0.33
W_POP = 0.33
W_UNC = 0.33
W_STOP = 0.33

gdf_vuln["vulnerability_score"] = (
    gdf_vuln["pop_norm"] * W_POP
    + gdf_vuln["unc_norm"] * W_UNC
    + gdf_vuln["bus_station_inv_norm"] * W_STOP
)

In [ ]:
# sort every small area by vulnerability score, most vulnerable first
# 취약 점수 기준으로 값이 큰 순서(가장 취약한 지역부터)로 전체 소지역을 정렬
ranking = gdf_vuln.sort_values("vulnerability_score", ascending=False)

#### From ranking to Top-20 / Top-5 / 랭킹에서 Top-20·Top-5로

With every small area scored, sorting by `vulnerability_score` (highest
first) turns the score into a ranking. `candidate_top20` is just that
ranking's first 20 rows -- not a separate selection step -- and
`final_top5` takes the first 5 of those. The two small areas used for the
rest of this tutorial (lecture 3's routing, lecture 4's 3D map) come
straight out of this ranking, whatever it happens to produce.

모든 소지역에 점수를 매겼으니, `vulnerability_score` 기준 내림차순 정렬이
곧 순위가 된다. `candidate_top20`은 그 순위의 상위 20행일 뿐 별도의 선별
단계가 아니고, `final_top5`는 그중 상위 5개를 취한다. 이 튜토리얼 뒤에서
쓰는 두 소지역(3강의 라우팅, 4강의 3D 지도)은 이 랭킹이 실제로 내놓는
결과를 그대로 따른 것이다.


In [ ]:
candidate_top20 = ranking[[
    "region_id", "region_nm", "ward_nm",
    "population", "pop_density",
    "bus_stop_cnt", "station_cnt",
    "uncovered_ratio", "vulnerability_score",
]]

candidate_top20.head(20)

In [ ]:
final_top5 = candidate_top20.head(5)
final_top5

In [ ]:
# the actual top-5 small areas, by region_id -- derived from the ranking
# above, not hand-picked, so this reflects whatever the real Hiroshima data
# produces
# 실제 취약도 상위 5개 소지역 -- 위 랭킹에서 직접 뽑은 것이지 임의로 고른
# 것이 아니라, 실제 히로시마 데이터가 만들어내는 결과 그대로다
top5_ids = final_top5["region_id"].astype(str).tolist()
gdf_top5 = gdf_vuln[gdf_vuln["region_id"].astype(str).isin(top5_ids)].copy()

In [ ]:
# A top-5 small area is a tiny sliver next to a whole city -- a thin outline
# would be invisible at city scale. Mark centroids instead, so the location
# is visible; the next cell zooms in for the actual shapes.
# 취약도 상위 5개 소지역은 도시 전체에 비하면 아주 작은 조각이라, 얇은
# 테두리로는 도시 스케일에서 보이지 않는다. 위치 확인용으로 중심점을
# 표시하고, 실제 모양은 다음 셀에서 확대해서 본다.
fig, ax = plt.subplots(figsize=(8, 8))

gdf_admin.plot(ax=ax, facecolor="none", edgecolor="lightgray", linewidth=0.3)

centroids = gdf_top5.geometry.representative_point()
ax.scatter(centroids.x, centroids.y, color="red", s=80, zorder=5, marker="*")

for _, row in gdf_top5.iterrows():
    p = row.geometry.representative_point()
    ax.annotate(
        row["region_nm"],
        (p.x, p.y),
        xytext=(8, 8),
        textcoords="offset points",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85),
    )

plt.title("Top-5 transit-vulnerable small areas (city-wide view)")
plt.axis("off")
plt.show()

In [ ]:
gdf_top5_uncovered = gpd.GeoDataFrame(
    gdf_top5.drop(columns=["geometry"]).rename(columns={"uncovered_geom": "geometry"}),
    crs=gdf_top5.crs,
)

# The top-5 areas aren't clustered together -- they land in two different,
# distant wards -- so one shared zoom window would either miss some of them
# or show mostly empty space between clusters. A small multiple (one tightly
# zoomed panel per area) works regardless of how scattered they are.
# 상위 5개 지역은 한데 모여 있지 않고 서로 먼 두 구에 나뉘어 있어서, 하나의
# 확대 범위로는 일부가 안 보이거나 클러스터 사이 빈 공간만 크게 보인다.
# 지역이 얼마나 흩어져 있든 상관없이, 지역마다 각자 딱 맞게 확대한 소형
# 패널을 나열하는 방식으로 그린다.
n = len(gdf_top5)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
if n == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, gdf_top5.iterrows()):
    minx, miny, maxx, maxy = row.geometry.bounds
    pad = max(maxx - minx, maxy - miny) * 0.3 or 50  # at least a little padding / 최소한의 여백 보장

    gdf_admin.plot(ax=ax, facecolor="none", edgecolor="lightgray", linewidth=0.5)
    gpd.GeoSeries([row.geometry], crs=gdf_top5.crs).plot(ax=ax, facecolor="none", edgecolor="red", linewidth=2)
    gdf_top5_uncovered[gdf_top5_uncovered["region_id"] == row["region_id"]].plot(
        ax=ax, color="red", alpha=0.3, edgecolor="none"
    )

    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)
    ax.set_title(row["region_nm"], fontsize=10)
    ax.axis("off")

plt.suptitle("Top-5 vulnerable areas, each zoomed to its own extent -- uncovered pockets shaded")
plt.tight_layout()
plt.show()

### Comparing Step 1 / Step 2 / Step 3 candidate sets
### Step 1 / Step 2 / Step 3 후보 비교

Three different candidate sets were produced along the way, each adding one
more consideration:

- **Step 1** (lecture 1): transit access alone -- the 10 small areas with
  the lowest combined bus + rail accessibility.
- **Step 2** (lecture 1): transit access + population density -- high
  density *and* poor bus access.
- **Step 3** (this lecture): transit + population + actual buffer coverage
  -- the vulnerability-score ranking.

Plotting all three together on one map makes it visible how much the
candidate set shifts (or doesn't) as more information is added.

지금까지 진행 순서대로 세 가지 다른 후보군이 나왔다. 각 단계는 고려 요소를
하나씩 더한다:

- **Step 1** (1강): 교통 접근성만 -- 버스+철도 접근성이 가장 낮은 10곳.
- **Step 2** (1강): 교통 접근성 + 인구밀도 -- 밀도는 높은데 버스 접근성은 낮은 곳.
- **Step 3** (이번 강): 교통 + 인구 + 실제 버퍼 커버리지 -- 취약 점수 랭킹.

세 후보군을 한 지도에 겹쳐 그리면, 정보를 더할 때마다 후보 집합이 얼마나
바뀌는지(혹은 안 바뀌는지) 눈으로 확인할 수 있다.

In [ ]:
# Step 1: small areas selected by transit access alone (from lecture 1)
# Step 1: 교통 접근성만으로 선정된 소지역 (1강)
stage1_names = target_top10["region_nm"].tolist()

# Step 2: transit + population density (from lecture 1)
# Step 2: 교통 + 인구밀도 (1강)
stage2_names = target2["region_nm"].tolist()

# Step 3: transit + population + coverage -- this lecture's final ranking
# Step 3: 교통 + 인구 + 커버리지 -- 이번 강의 최종 랭킹
stage3_names = candidate_top20.head(10)["region_nm"].tolist()

In [ ]:
gdf_stage1 = gdf_admin[gdf_admin["region_nm"].isin(stage1_names)].copy()
gdf_stage2 = gdf_admin[gdf_admin["region_nm"].isin(stage2_names)].copy()
gdf_stage3 = gdf_admin[gdf_admin["region_nm"].isin(stage3_names)].copy()

In [ ]:
import folium
from folium.plugins import MarkerCluster

# reproject everything to WGS84 for Folium
# Folium용으로 전부 WGS84(위경도)로 변환
gdf_admin_wgs  = gdf_admin.to_crs(epsg=4326)
gdf_stage1_wgs = gdf_stage1.to_crs(epsg=4326)
gdf_stage2_wgs = gdf_stage2.to_crs(epsg=4326)
gdf_stage3_wgs = gdf_stage3.to_crs(epsg=4326)
gdf_station_wgs = gdf_station.to_crs(epsg=4326)
gdf_bus_wgs    = gdf_bus.to_crs(epsg=4326)

# centre the map on Hiroshima City's actual extent, not a hardcoded point
# 하드코딩된 좌표가 아니라 히로시마시 실제 범위 중심으로 지도를 잡는다
center = gdf_admin_wgs.geometry.union_all().centroid

m = folium.Map(location=[center.y, center.x], zoom_start=11, tiles=None)

folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",
    attr="\u00a9 OpenStreetMap contributors",
    name="OpenStreetMap",
    overlay=False,
    control=True,
    max_zoom=19,
).add_to(m)

# all small areas, faint background
# 전체 소지역, 연한 배경
folium.GeoJson(
    gdf_admin_wgs,
    name="All small areas",
    style_function=lambda x: {"fillColor": "#aaaaaa", "fillOpacity": 0.05, "color": "#888888", "weight": 0.8},
).add_to(m)

# Step 1: transit only (dashed blue)
# Step 1: 교통만 (파란 점선)
folium.GeoJson(
    gdf_stage1_wgs,
    name="Step 1: transit only",
    style_function=lambda x: {"fillColor": "#4a90d9", "fillOpacity": 0.15, "color": "#1a5fb4", "weight": 2.5, "dashArray": "8 5"},
).add_to(m)

# Step 2: transit + population (dashed orange)
# Step 2: 교통 + 인구 (주황 점선)
folium.GeoJson(
    gdf_stage2_wgs,
    name="Step 2: transit + population",
    style_function=lambda x: {"fillColor": "#f5a623", "fillOpacity": 0.2, "color": "#d07000", "weight": 2.5, "dashArray": "8 5"},
).add_to(m)

# Step 3: transit + population + coverage (solid red, most emphasized)
# Step 3: 교통 + 인구 + 커버리지 (빨간 실선, 가장 강조)
folium.GeoJson(
    gdf_stage3_wgs,
    name="Step 3: transit + population + coverage",
    style_function=lambda x: {"fillColor": "#e74c3c", "fillOpacity": 0.25, "color": "#c0392b", "weight": 3.5},
).add_to(m)

# station markers
# 역 마커
station_group = folium.FeatureGroup(name="Stations")
station_group.add_to(m)
for _, row in gdf_station_wgs.iterrows():
    if row.geometry is None or row.geometry.is_empty:
        continue
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=6, color="#1a5fb4", fill=True, fill_color="#4a90d9",
        fill_opacity=0.9, weight=2,
        tooltip=str(row.get("N02_005", "station")),
    ).add_to(station_group)

# bus stop cluster markers
# 버스정류장 클러스터 마커
cluster = MarkerCluster(name="Bus stops")
cluster.add_to(m)
for _, row in gdf_bus_wgs.iterrows():
    if row.geometry is None or row.geometry.is_empty:
        continue
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4, color="#27ae60", fill=True, fill_color="#2ecc71",
        fill_opacity=0.85, weight=1.5,
        tooltip=str(row.get("P11_001", "bus stop")),
    ).add_to(cluster)

folium.LayerControl(collapsed=False).add_to(m)
m.save(os.path.join(OUT_DIR, "hiroshima_steps_compare.html"))
m

### [Limitations] Insights from comparing vulnerability across stages
### [과업의 한계] 단계별 교통 취약 지역 비교 인사이트

The same Hiroshima City data was run through three stages -- (1) transit
alone, (2) transit + population, (3) transit + population + straight-buffer
coverage -- to compare which small areas come out as vulnerable. Each stage
isn't just adding another indicator; it's a check on whether that indicator
actually adds new information.

같은 히로시마시 데이터에 ① 교통만, ② 교통 + 인구, ③ 교통 + 인구 + 직선
버퍼 커버리지를 순차적으로 적용해 취약 지역을 비교했다. 각 단계는 지표를
단순히 추가하는 것이 아니라, 그 지표가 실제로 새로운 정보를 더하는지
검증하는 과정이다.

#### Step 1: transit only
- Counting stops alone tends to flag low-density fringe areas as vulnerable
- Some of those see little real ridership demand -- overstating vulnerability
- Supply (infrastructure count) alone isn't enough to set policy priority

#### Step 1: 교통만 기준
- 정류장 개수만 보면 저밀도 외곽 지역이 취약 지역으로 다수 선정됨
- 실제 이용 수요가 거의 없는 지역까지 취약으로 과대평가될 수 있음
- 공급량(인프라 수)만으로는 정책 우선순위를 정하기 어려움

#### Step 2: transit + population
- Adding population shifts the candidates toward dense residential areas
- "Few stops, high demand" areas surface -- more policy-relevant
- Supply and demand together give a different vulnerable-area distribution

#### Step 2: 교통 + 인구 기준
- 인구를 함께 고려하면 취약 지역이 주거 밀집 지역 중심으로 이동
- "정류장이 적으면서 수요가 큰 지역"이 선별되어 정책적 의미가 증가
- 공급과 수요를 함께 고려하면 취약지역 분포가 달라짐

#### Step 3: transit + population + buffer coverage
- Adding buffer coverage corrects for walking distance to a stop
- But coverage correlates strongly with stop count itself: more stops ->
  wider coverage, fewer stops -> narrower coverage, almost by construction
- The result is **spatial multicollinearity**: an indicator that looks new
  but largely repeats the spatial pattern an existing one already captured
- Straight-line buffer distance alone doesn't explain real accessibility
  differences

#### Step 3: 교통 + 인구 + 버퍼 커버리지 기준
- 버퍼 커버리지를 추가하면 도보 거리를 보정할 수 있음
- 그러나 커버리지는 정류장 수 자체와 강한 상관관계를 가짐: 정류장이 많은
  지역은 커버도 넓고, 적은 지역은 커버도 좁다는 구조가 거의 그대로 반복됨
- 결과적으로 **공간적 다중공선성**(spatial multicollinearity) 문제가
  발생한다: 새로운 지표처럼 보이지만 기존 지표와 거의 같은 공간 패턴을
  반복하는 현상
- 직선거리 기반 버퍼만으로는 실제 접근성 차이를 충분히 설명하기 어렵다

### Summary and next step
- Which areas count as "vulnerable" depends heavily on how the indicator is designed
- Stop count and straight-line coverage share the same underlying spatial structure, so their explanatory power overlaps
- Straight-line accessibility in particular doesn't reflect real road networks, dead ends, or detours
- A more realistic accessibility analysis needs actual walking routes, not just a radius
- The next lecture extends this by incorporating real pedestrian-network routing

### 종합 해석 및 다음 단계
- 취약 지역은 지표 설계 방식에 따라 크게 달라질 수 있다
- 정류장 수와 직선거리 기반 커버리지는 같은 공간 구조를 공유해 설명력이 중복될 가능성이 있다
- 특히 직선거리 기반 접근성은 실제 도로 구조, 단절 구간, 우회 경로를 반영하지 못한다
- 보다 현실적인 접근성 분석을 위해서는 단순 반경이 아니라 실제 이동 경로를 반영해야 한다
- 다음 강의에서는 실제 보행 네트워크 경로를 고려한 분석으로 확장한다

### Spatial multicollinearity
### 공간적 다중공선성 (Spatial Multicollinearity)

**Spatial multicollinearity** is when two variables look different but
share almost the same spatial distribution pattern, so in practice they
carry redundant information.

**공간적 다중공선성**이란, 서로 다른 변수처럼 보이지만 공간적으로 거의
동일한 분포 패턴을 공유해 실질적으로는 중복된 정보를 제공하는 현상을
의미한다.

That is: areas with more stops also show more coverage, and areas with
fewer stops show less coverage -- the same spatial structure repeating
across two nominally different indicators. When that happens, adding the
second indicator doesn't meaningfully increase explanatory power, and it
limits how well the indicators can actually distinguish vulnerable areas.

즉, 정류장 수가 많은 지역은 커버리지도 높게 나타나고, 정류장 수가 적은
지역은 커버리지도 낮게 나타나는 경우처럼, 두 지표가 같은 공간 구조를
반복할 때 발생한다. 이 경우 지표를 추가해도 설명력이 크게 늘지 않으며,
취약지역을 구분하는 힘이 제한될 수 있다.

In [ ]:
# select the demo small area(s) for the next lecture -- derived directly
# from the ranking above, not hand-picked, so this reflects whatever the
# real Hiroshima data produces.
# 다음 강의에서 쓸 데모 소지역 -- 위 랭킹에서 직접 뽑은 것이지 임의로 고른
# 것이 아니다. 실제 히로시마 데이터가 만들어내는 결과를 그대로 반영한다.
final_names = candidate_top20.head(2)["region_nm"].tolist()
print("demo small areas for the next lecture:", final_names)

gdf_demo = gdf_vuln[gdf_vuln["region_nm"].isin(final_names)].copy()
gdf_demo["region_id"] = gdf_demo["region_id"].astype(str)

os.makedirs(OUT_DIR, exist_ok=True)

# 1) the small-area polygons themselves
# 1) 소지역 폴리곤 자체
gdf_demo_admin = gdf_demo.drop(columns=["uncovered_geom"]) if "uncovered_geom" in gdf_demo.columns else gdf_demo.copy()

OUT_GPKG_ADMIN = os.path.join(OUT_DIR, "demo_admin.gpkg")
if os.path.exists(OUT_GPKG_ADMIN):
    os.remove(OUT_GPKG_ADMIN)
gdf_demo_admin.to_file(OUT_GPKG_ADMIN, layer="admin", driver="GPKG")
print("saved small-area polygons:", OUT_GPKG_ADMIN)

# 2) just the uncovered polygons (the difference-operation result)
# 2) 비커버 폴리곤만 (차집합 연산 결과)
gdf_demo_uncovered = gpd.GeoDataFrame(
    gdf_demo.drop(columns=["geometry"]).rename(columns={"uncovered_geom": "geometry"}),
    crs=gdf_demo.crs,
)

OUT_GPKG_UNC = os.path.join(OUT_DIR, "demo_uncovered.gpkg")
if os.path.exists(OUT_GPKG_UNC):
    os.remove(OUT_GPKG_UNC)
gdf_demo_uncovered.to_file(OUT_GPKG_UNC, layer="uncovered", driver="GPKG")
print("saved uncovered polygons:", OUT_GPKG_UNC)


### Lecture 2 summary / 2강 요약

**Key takeaways / 핵심 요약**
- Coverage is exact geometry (`.difference()`), not a yes/no test -- see the spatial-multicollinearity discussion above for why the final ranking still needs care even with exact geometry.
- A ranking built from equal weights (0.33/0.33/0.33) is a *choice*, not a fact -- try the exercise below with different weights and see how much the Top-5 actually moves.

**핵심 요약**
- 커버리지는 예/아니오 판정이 아니라 정확한 지오메트리 연산(`.difference()`)이다 -- 그럼에도 최종 랭킹에 왜 여전히 주의가 필요한지는 위 공간적 다중공선성 논의 참고.
- 동일 가중치(0.33/0.33/0.33)로 만든 순위는 사실이 아니라 *선택*이다 -- 아래 실습에서 다른 가중치로 직접 돌려보고 Top-5가 얼마나 바뀌는지 확인해보자.

**Real-world tip / 실무 팁**
This lecture's `.difference()` approach is the correct pattern -- but the project's own `analysis/lecture4_grid_bias.py` forensic script had, at one point, computed the same kind of "is this cell uncovered" question with a binary `.intersects()` instead, overstating uncovered population by **+51.7%**. Same city, same data, same *kind* of question -- one bit of code asked "does this cell touch the uncovered area at all," the other asked "how much of it," and the answers diverged by half. When you write `.intersects()` for anything other than a true yes/no filter, ask whether the real question is actually "how much," not "whether."

**실무 팁**
이 강의 `.difference()` 방식이 올바른 패턴이다 -- 그런데 이 프로젝트 자체의 포렌식 스크립트 `analysis/lecture4_grid_bias.py`는 한때 "이 격자가 비커버인가"라는 같은 종류의 질문을 이진 `.intersects()`로 계산해서 비커버 인구를 **+51.7%** 과대계상한 적이 있다. 같은 도시, 같은 데이터, 같은 *종류*의 질문인데 -- 한쪽은 "이 격자가 비커버 영역에 조금이라도 닿는가"를, 다른 쪽은 "얼마나 닿는가"를 물었고 답이 절반 가까이 갈렸다. 진짜 예/아니오 필터가 아닌 곳에 `.intersects()`를 쓰고 있다면, 진짜 질문이 "닿는지 여부"가 아니라 "얼마나"인지 다시 확인하자.

**Try it yourself / 직접 해보기**
Insert a new cell and recompute the vulnerability ranking with different weights, without touching `gdf_vuln`. How many of the original Top-10 areas are still in your new Top-10?

새 셀을 추가해서 `gdf_vuln`은 건드리지 않고 다른 가중치로 취약도 순위를 다시 계산해보자. 원래 Top-10 중 몇 곳이 새 Top-10에도 남아있는가?

<details>
<summary>Show one way to do it / 정답 예시 보기</summary>

```python
my_score = (
    gdf_vuln["pop_norm"] * 0.5
    + gdf_vuln["unc_norm"] * 0.3
    + gdf_vuln["bus_station_inv_norm"] * 0.2
)
my_ranking = gdf_vuln.assign(my_score=my_score).sort_values("my_score", ascending=False)
my_ranking[["region_nm", "my_score"]].head(10)
```

</details>


### [Lecture 3] How far do residents actually have to walk?
### [3강] 실제로 얼마나 멀리 걸어야 하는가

Lecture 2 measured coverage with straight-line buffers (300m bus / 500m
rail) and found the two most vulnerable small areas -- 海老園四丁目 and
海老山南一丁目 -- **0% covered**: not a single stop falls within a
straight-line buffer of either polygon. But a straight-line buffer isn't
a route -- it ignores roads, dead ends, slopes, and detours. Lecture 3
checks the same question against the real pedestrian network instead:
starting from inside these two small areas, how far do you actually have
to walk, along real streets, to reach the nearest bus stop or station?

2강은 직선 버퍼(버스 300m/철도 500m)로 커버리지를 측정했고, 가장 취약한
두 소지역 海老園四丁目·海老山南一丁目은 **커버율 0%**였다: 두 폴리곤 중
어디에도 직선 버퍼 안에 정류장이 하나도 없다. 그러나 직선 버퍼는 실제
경로가 아니다 -- 도로, 막다른 길, 경사, 우회를 반영하지 못한다. 3강은
같은 질문을 실제 보행 네트워크로 다시 확인한다: 이 두 소지역 안에서
출발했을 때, 실제 도로를 따라 가장 가까운 버스정류장이나 역까지 얼마나
걸어야 하는가?

*A note on data: a natural routing origin here would be Hiroshima's
bike-share stations, ぴーすくる (Peesakuru) -- "how far does bike-share
reach." But its station data is only published through
公共交通オープンデータセンター (ODPT), which requires a free developer
account to register for separately. Rather than block this section on
that, the routing origin here is a sample grid of points inside the two
vulnerable small areas themselves -- "starting from where people actually
live" is arguably a more direct question than "starting from a
bike-share dock" anyway.*

*데이터 참고: 자연스러운 라우팅 출발점은 히로시마의 자전거 공유
서비스인 ぴーすくる일 것이다("자전거 공유가 어디까지 닿는가"). 하지만
정류소 데이터는 公共交通オープンデータセンター(ODPT)를 통해서만 제공되는데,
별도로 무료 개발자 계정을 등록해야 한다. 이 때문에 막히기보다, 여기서는
취약 소지역 두 곳 내부에 표본 격자점을 만들어 출발점으로 쓴다 -- "자전거
공유 정류소에서 출발"보다 "실제 주민이 사는 곳에서 출발"이 오히려 더
직접적인 질문이기도 하다.*

The straight-line "0% covered" figure above isn't wrong, but "0%
covered" and "unreachable" aren't the same claim. Pushing on that gap --
straight-line distance versus an actual walkable route -- is what the
rest of this lecture does.

위의 직선거리 기준 "커버율 0%"는 틀린 숫자가 아니지만, "커버율 0%"와
"닿을 수 없다"가 같은 말은 아니다. 직선거리와 실제로 걸을 수 있는
경로 사이의 그 간극을 파고드는 게 이 강의 나머지가 하는 일이다.


#### OSMnx / NetworkX shortest-path routing
#### OSMnx · NetworkX 기반 최단경로 알고리즘

**OSMnx** builds road/footpath/cycleway networks as graph structures from
OpenStreetMap's global open road data. **NetworkX** is a general graph-
theory engine that runs shortest-path and weighted-distance calculations
on top of the graph OSMnx builds. Used together, they compute the real
routable shortest path along actual streets, not a straight line.

**OSMnx**는 OpenStreetMap이 제공하는 전 세계 공개 도로 데이터를 기반으로
도로·보행로·자전거도로 네트워크를 그래프 구조로 만들어주는 라이브러리다.
**NetworkX**는 그래프 이론 분석 엔진으로, OSMnx가 만든 그래프 위에서
최단경로·가중치 기반 거리 계산을 수행한다. 두 라이브러리를 함께 쓰면
직선거리가 아니라 실제 도로 위의 최단경로를 계산할 수 있다.

In [ ]:
# Not run here: osmnx is already installed in this notebook's uv
# environment (see cell 2's note); %pip install would fail loudly for no
# reason in this venv.
# 여기서는 실행하지 않음: osmnx는 이 노트북의 uv 환경에 이미
# 설치되어 있다(셀 2의 설명 참고). 이 venv에서는 %pip install이 굳이
# 큰 소리로 실패한다.
# %pip install osmnx


In [ ]:
import os
import warnings

# Do not suppress GIS warnings globally; they often reveal CRS or geometry problems.
warnings.filterwarnings("default")

#### 0. Data directory setup / 0. 데이터 디렉토리 설정

In [ ]:
# Reuse the repository paths established in Lecture 1.
BASE_DIR = Path(BASE_DIR)
DATA_DIR = BASE_DIR / "data" / "hiroshima"
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

#### 1. Load data / 1. 데이터 로드

In [ ]:
# the two most vulnerable small areas, saved at the end of lecture 2
# 2강 마지막에 저장한, 가장 취약한 두 소지역
ADMIN_GPKG = os.path.join(OUT_DIR, "demo_admin.gpkg")
gdf_admin = gpd.read_file(ADMIN_GPKG).to_crs(6670)

UNCOVERED_GPKG = os.path.join(OUT_DIR, "demo_uncovered.gpkg")
gdf_unc = gpd.read_file(UNCOVERED_GPKG).to_crs(6670)

gdf_admin[["region_nm", "ward_nm", "population"]]

In [ ]:
# reload city-wide bus stops and stations, same sources lecture 1 used
# 1강에서 쓴 것과 같은 시내 전역 버스정류장·역 데이터를 다시 불러온다
gdf_bus = gpd.read_file(os.path.join(DATA_DIR, "hiroshima_city_bus_stops.gpkg")).to_crs(6670)
gdf_bus["stop_nm"] = gdf_bus["P11_001"]

gdf_station = gpd.read_file(os.path.join(DATA_DIR, "hiroshima_city_stations.gpkg")).to_crs(6670)
gdf_station["geometry"] = gdf_station.geometry.centroid  # platform LineString -> point, as in lecture 1
                                                            # 플랫폼 선분 -> 점, 1강과 동일 처리
gdf_station["stop_nm"] = gdf_station["N02_005"]

#### Choosing a Folium basemap tile
#### Folium 지도(Tile) 선택 기준 이해하기
- The map tile isn't just background -- it directly affects how readable the analysis result is.
- So instead of picking "whatever tile," get in the habit of choosing deliberately for the purpose at hand.

- 지도 타일은 단순한 배경이 아니라 분석 결과의 가독성에 직접 영향을 준다.
- 따라서 "아무 타일이나" 쓰지 말고, 목적에 맞는 타일을 의도적으로 선택하는 습관이 필요하다.

##### For hands-on practice, prioritize reliability
##### 실습이 목적일 때는 안정성을 최우선으로 선택한다
- If a tile fails to load in your environment (VS Code, Jupyter, Colab...), it breaks the flow of the exercise.
- `tiles="OpenStreetMap"` is a safe default for practice/homework/early drafts.
- 실습 환경에 따라 타일이 안 뜨면 분석 흐름이 끊긴다. `tiles="OpenStreetMap"`을 초기 작성 단계의 기본값으로 쓴다.

##### When presenting results, favour legibility
##### 결과를 보여주는 단계에서는 지도 가독성을 고려한다
- After the analysis is done, viewers need to see "what was analyzed" at a glance.
- `tiles="CartoDB Voyager"` suits shared results, talks, and report figures.
- 분석이 끝난 뒤에는 "무엇을 분석했는지"가 한눈에 보여야 한다. `tiles="CartoDB Voyager"`가 공유·발표용에 적합하다.

##### When the analysis result itself is the point, simplify the background
##### 분석 결과 자체를 설명해야 할 때는 배경을 최대한 단순화한다
- For vulnerable-area / uncovered-area / cluster maps, a busy background competes with the result.
- `tiles="CartoDB Positron"` suits policy interpretation and result comparison.
- 취약지역·비커버지역·클러스터처럼 결과가 핵심일 때는 배경이 복잡하면 해석을 방해한다. `tiles="CartoDB Positron"`이 적합하다.

In [ ]:
# Assign points robustly; exact boundary points get one deterministic region.
gdf_bus_sel = assign_points_to_regions(
    gdf_bus[["stop_nm", "geometry"]], gdf_admin[["region_nm", "geometry"]],
    id_col="region_nm", name_col="region_nm",
)
gdf_bus_sel = gdf_bus_sel.dropna(subset=["region_nm"]).drop(
    columns=["index_right", "_point_id", "boundary_dist_m"], errors="ignore"
)
gdf_bus_sel

#### Why Folium visualization uses EPSG:4326
#### Folium 시각화 시 EPSG:4326을 사용하는 이유

Folium runs on Leaflet.js internally, and Leaflet interprets every spatial
object as longitude/latitude -- EPSG:4326 (WGS84). A projected CRS (like
EPSG:6670, in metres) handed to Folium directly gets misread, and objects
end up in the wrong place or don't render at all. The basemap tiles Folium
uses (OpenStreetMap, CartoDB, ...) all assume WGS84 input too, so your
data has to match to line up correctly.

Folium은 내부적으로 Leaflet.js로 동작하며, Leaflet은 모든 공간 객체를
경도·위도(EPSG:4326, WGS84)로 해석한다. 투영좌표계(EPSG:6670 등, 단위:
미터)를 그대로 넘기면 좌표가 잘못 해석되어 객체가 엉뚱한 위치에 표시되거나
아예 보이지 않는다. Folium이 쓰는 베이스맵 타일(OpenStreetMap, CartoDB 등)도
전부 WGS84 입력을 전제로 하므로, 사용자 데이터도 EPSG:4326으로 맞춰야
정확히 겹쳐진다.

##### Practical rule
##### 실무 기준 좌표계 사용 원칙
- **Spatial analysis stage**: EPSG:6670 (metric, for distance/area/buffer/coverage)
- **Visualization stage (Folium)**: EPSG:4326 (WGS84, the Leaflet web-map standard)
- **공간 분석 단계**: EPSG:6670 (미터 단위, 거리·면적·버퍼·커버리지 분석용)
- **지도 시각화 단계(Folium)**: EPSG:4326 (WGS84, Leaflet 기반 웹 지도 표준)

In [ ]:
admin_ll = gdf_admin.to_crs(4326)
unc_ll   = gdf_unc.to_crs(4326)
bus_ll   = gdf_bus_sel.to_crs(4326)

bounds = admin_ll.total_bounds
center_lat = (bounds[1] + bounds[3]) / 2
center_lon = (bounds[0] + bounds[2]) / 2

In [ ]:
m = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles="OpenStreetMap")

# 1) small-area boundary
# 1) 소지역 경계
folium.GeoJson(
    admin_ll, name="small areas / 소지역",
    style_function=lambda x: {"fillOpacity": 0.05, "color": "#666666", "weight": 2},
    tooltip=folium.GeoJsonTooltip(fields=["region_nm"], aliases=["area / 지역"]),
).add_to(m)

# 2) uncovered area
# 2) 비커버 지역
folium.GeoJson(
    unc_ll, name="uncovered / 비커버 지역",
    style_function=lambda x: {"fillOpacity": 0.30, "color": "#cc0000", "weight": 1},
    tooltip=folium.GeoJsonTooltip(fields=["region_nm"], aliases=["area / 지역"]),
).add_to(m)

# 3) bus stops (clustered)
# 3) 버스정류장 (클러스터)
fg_bus = folium.FeatureGroup(name="bus stops / 버스정류장", show=True)
mc = MarkerCluster().add_to(fg_bus)
for _, r in bus_ll.iterrows():
    folium.Marker(
        location=[r.geometry.y, r.geometry.x],
        tooltip=f"bus / 버스 | {r.get('stop_nm','')}",
        icon=folium.Icon(color="blue", icon="bus", prefix="fa"),
    ).add_to(mc)
fg_bus.add_to(m)

m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
folium.LayerControl(collapsed=False).add_to(m)

OUT_HTML = os.path.join(OUT_DIR, "admin_uncovered_bus.html")
m.save(OUT_HTML)
OUT_HTML

#### Folium core functions at a glance
#### Folium 핵심 함수 한눈 요약

Folium is a web-map library that works as `create map -> add layers -> control`.

Folium은 `지도 생성 → 레이어 추가 → 제어` 구조로 동작하는 웹 지도 라이브러리다.

- **`folium.Map()`**: the map's starting point -- centre, zoom, basemap
- **`folium.GeoJson()`**: renders spatial objects (polygons/lines) like boundaries or analysis regions
- **`style_function`**: colour/width/opacity styling for a GeoJson layer
- **`folium.GeoJsonTooltip()`**: attribute info shown on hover
- **`folium.FeatureGroup()`**: bundles several objects into one togglable layer
- **`folium.Marker()`**: a single point marker
- **`folium.Icon()`**: marker icon shape/colour (FontAwesome-backed)
- **`MarkerCluster`**: auto-clusters many markers
- **`fit_bounds()`**: auto-fits the view to show everything
- **`folium.LayerControl()`**: a UI to toggle/compare layers
- **`save()`**: writes the map to an HTML file

- **`folium.Map()`**: 지도의 시작점 -- 중심, 줌, 베이스맵
- **`folium.GeoJson()`**: 경계·분석영역 같은 공간 객체(폴리곤/라인) 시각화
- **`style_function`**: GeoJson 레이어의 색상·두께·투명도 지정
- **`folium.GeoJsonTooltip()`**: 마우스 오버 시 속성 정보 표시
- **`folium.FeatureGroup()`**: 여러 객체를 하나의 켜고 끌 수 있는 레이어로 묶음
- **`folium.Marker()`**: 개별 지점 표시
- **`folium.Icon()`**: 마커 아이콘 모양·색상 (FontAwesome 지원)
- **`MarkerCluster`**: 다수 마커 자동 군집화
- **`fit_bounds()`**: 모든 객체가 보이도록 범위 자동 조정
- **`folium.LayerControl()`**: 레이어 선택·비교 UI
- **`save()`**: 지도를 HTML 파일로 저장

Folium documentation: https://python-visualization.github.io/folium/latest/

In [ ]:
gdf_station_sel = assign_points_to_regions(
    gdf_station[["stop_nm", "geometry"]],
    gdf_admin[["region_nm", "geometry"]],
    id_col="region_nm", name_col="region_nm",
)
gdf_station_sel = gdf_station_sel.dropna(subset=["region_nm"]).drop(
    columns=["index_right", "_point_id", "boundary_dist_m"], errors="ignore"
)

print(f"stations inside the small areas themselves: {len(gdf_station_sel)}")
print(f"소지역 내부에 있는 역: {len(gdf_station_sel)}개")

#### 3. OSMnx / NetworkX shortest-path routing
#### 3. OSMnx · NetworkX 기반 최단경로 알고리즘

The goal of this part: **starting from inside the two vulnerable small
areas, compute the real walking route -- and how long it actually takes --
to every bus stop or station reachable within a 15-minute walk**. Not
straight-line distance; distance along the actual pedestrian network.

이 파트의 목적: **취약 소지역 두 곳 내부에서 출발해, 실제 도로(보행
네트워크)를 따라 15분 이내에 도달 가능한 모든 버스정류장·역까지의 경로와
소요 시간을 계산하는 것**이다. 직선거리가 아니라 실제 보행 네트워크
위의 거리를 기준으로 한다.

First, merge the small-area polygons into one and buffer it, so the
network doesn't cut off right at the boundary. Reproject to EPSG:4326
(OSMnx's expected CRS) and build the walk network with
`graph_from_polygon()`.

먼저 소지역 폴리곤을 하나로 합치고 버퍼를 적용해, 경계에서 네트워크가
끊기지 않게 한다. OSMnx가 쓰는 EPSG:4326으로 변환한 뒤
`graph_from_polygon()`으로 보행 네트워크를 만든다.

Origins and destinations are point coordinates, but NetworkX routing works
on graph node IDs. `nearest_nodes()` snaps each point to its nearest graph
node. Several stops can snap to the same node, so a node -> stops mapping
prevents any of them from being silently dropped.

출발지·목적지는 좌표(Point)지만 NetworkX 라우팅은 그래프 노드 ID
기준으로 동작한다. `nearest_nodes()`로 각 지점을 가장 가까운 노드에
스냅한다. 여러 정류장이 같은 노드에 스냅될 수 있으므로, 노드→정류장
매핑을 만들어 결과 누락을 방지한다.

For each origin, `single_source_dijkstra_path_length()` first finds every
node reachable within the 15-minute cutoff -- this avoids the waste of
computing a full route to every single stop in the city. Only the stop
nodes that fall within that reachable set get an actual route rebuilt with
`shortest_path()`.

각 출발점에 대해 `single_source_dijkstra_path_length()`로 15분 컷오프
이내에 도달 가능한 모든 노드를 먼저 계산한다 -- 도시 전체 정류장까지
매번 전체 경로를 계산하는 낭비를 피하기 위함이다. 그 도달 범위 안에 든
정류장 노드에 대해서만 `shortest_path()`로 실제 경로를 복원한다.

The rebuilt route is a LineString; its length is computed in a metric
projected CRS (EPSG:6670), not lat/lon degrees.

복원된 경로는 LineString이며, 길이는 위경도(도) 단위가 아니라 미터
단위 투영좌표계(EPSG:6670)에서 계산한다.

#### Heads up: this downloads real OSM data / 참고: 실제 OSM 데이터를 내려받는다

The next few cells fetch the walking network from OpenStreetMap over the
internet (`graph_from_polygon`) -- typically under a minute for an area
this size, but slower on conference wifi. Overpass responses are cached
under `cache/` once fetched, so a second run (e.g. a re-run during Q&A)
is instant. If this seems to hang for more than ~2 minutes, check your
internet connection before assuming something is broken.

다음 몇 셀은 실제로 인터넷을 통해 OpenStreetMap에서 보행 네트워크를
받아온다(`graph_from_polygon`) -- 이 정도 면적이면 보통 1분 이내지만
컨퍼런스 wifi에서는 더 걸릴 수 있다. 한 번 받으면 `cache/`에 캐시되므로
(예: Q&A 중 재실행) 두 번째부터는 즉시 끝난다. 2분 넘게 멈춘 것처럼
보이면, 뭔가 고장났다고 판단하기 전에 인터넷 연결부터 확인하자.


In [ ]:
# OSMnx needs one polygon enclosing the whole analysis area. If the small
# areas were left separate, the network could cut off or fail to route near
# the boundary, so merge them and add a generous buffer.
# OSMnx는 분석 대상을 감싸는 하나의 폴리곤이 필요하다. 소지역을 따로 두면
# 경계 근처에서 네트워크가 끊기거나 라우팅이 실패할 수 있어, 하나로 합치고
# 충분한 버퍼를 준다.
poly_union_6670 = gdf_admin.geometry.union_all().buffer(0)
# buffer(0) is the usual trick to fix minor geometry glitches
# buffer(0)은 미세한 지오메트리 오류를 정정하는 관용적 처리

poly_buf_6670 = poly_union_6670.buffer(2000)  # 2km margin around the small areas / 소지역 주변 2km 여유
poly_buf_4326 = gpd.GeoSeries([poly_buf_6670], crs=6670).to_crs(4326).iloc[0]

In [ ]:
ox.settings.log_console = False
ox.settings.use_cache = True
ox.settings.cache_folder = os.path.join(BASE_DIR, "cache")

G = ox.graph_from_polygon(poly_buf_4326, network_type="walk", simplify=True)
print(f"network nodes: {G.number_of_nodes()}, edges: {G.number_of_edges()}")

In [ ]:
# combine bus + station into one stop table, tagged by type, for uniform routing
# 버스+역을 유형 태그를 붙여 하나의 정류장 테이블로 통합 (라우팅을 동일 로직으로 처리)
gdf_bus_tagged = gdf_bus.copy()
gdf_bus_tagged["stop_type"] = "bus"

gdf_station_tagged = gdf_station.copy()
gdf_station_tagged["stop_type"] = "station"

gdf_stop_6670 = pd.concat(
    [gdf_bus_tagged[["stop_nm", "stop_type", "geometry"]],
     gdf_station_tagged[["stop_nm", "stop_type", "geometry"]]],
    ignore_index=True,
)
gdf_stop_6670 = gpd.GeoDataFrame(gdf_stop_6670, geometry="geometry", crs=6670)
gdf_stop_ll = gdf_stop_6670.to_crs(4326)  # OSMnx/NetworkX need WGS84 / OSMnx·NetworkX는 WGS84 필요

len(gdf_stop_ll)

#### Nodes and links
#### 노드(Node)와 링크(Link)

Network analysis represents space as a **graph**.

네트워크 분석은 공간을 **그래프 구조**로 표현한다.

- **Node**: where movement starts or ends (intersections, road ends, where a stop snaps to)
- **Link (Edge)**: a traversable connection between two nodes (a road, footpath, cycleway)

- **노드(Node)**: 이동이 시작·종료되는 지점 (교차로, 도로 끝, 정류소가 매칭되는 위치)
- **링크(Link/Edge)**: 두 노드를 연결하는 이동 가능한 경로 (도로, 보행로, 자전거도로)

Routing and distance calculations run on the weights (distance, time) of
the links between nodes. Origins and destinations must be snapped to
nodes first -- network analysis only operates on the node-link structure.

경로 탐색과 거리 계산은 노드 간 링크의 가중치(거리·시간)를 기준으로
수행된다. 출발지·목적지는 반드시 노드에 먼저 스냅되어야 하며, 네트워크
분석은 노드-링크 구조 위에서만 가능하다.

In [ ]:
try:
    stop_nodes = ox.distance.nearest_nodes(
        G, X=gdf_stop_ll.geometry.x.values, Y=gdf_stop_ll.geometry.y.values,
    )
    gdf_stop_ll["v_node"] = stop_nodes
except Exception as e:
    raise RuntimeError(f"stop snapping failed / 정류장 스냅 실패: {e}")

In [ ]:
# several nearby stops commonly snap to the same graph node -- keep a
# node -> [stop indices] map so none of them get lost
# 서로 가까운 정류장이 같은 그래프 노드에 매칭되는 경우가 흔함 --
# 노드 -> [정류장 인덱스] 매핑으로 누락을 방지한다
node_to_stops = {}
for i, row in gdf_stop_ll.iterrows():
    node_to_stops.setdefault(row["v_node"], []).append(i)

stop_node_set = set(node_to_stops.keys())
len(stop_node_set)

In [ ]:
# Sample a regular grid plus a representative point for every polygon.
# The representative point prevents small/thin polygons from producing zero origins.
def grid_points(poly, spacing=100):
    minx, miny, maxx, maxy = poly.bounds
    xs = np.arange(np.floor(minx / spacing) * spacing, maxx + spacing, spacing)
    ys = np.arange(np.floor(miny / spacing) * spacing, maxy + spacing, spacing)
    points = [Point(x, y) for x in xs for y in ys if poly.covers(Point(x, y))]
    representative = poly.representative_point()
    if not points or all(p.distance(representative) > spacing / 2 for p in points):
        points.append(representative)
    return points


origins = []
for _, arow in gdf_admin.iterrows():
    for point in grid_points(arow.geometry, spacing=100):
        origins.append((arow["region_nm"], point))

print(f"origin points: {len(origins)}")

In [ ]:
WALK_SPEED_KMH = 4.8  # a standard walking-speed assumption / 표준 도보 속도 가정
TIME_MIN = 15          # walk-tolerance threshold for this analysis / 이 분석의 도보 허용 기준

# 4.8 km/h -> 80 m/min -> 15 min = 1,200 m
CUTOFF_M = WALK_SPEED_KMH * 1000 / 60 * TIME_MIN
print(f"cutoff distance: {CUTOFF_M:.0f}m")

routes = []  # successful routes / 성공한 경로
fails = []   # routing failures, for transparency / 실패 사례 (투명성을 위해 기록)

In [ ]:
for region_nm, origin_pt_6670 in origins:
    origin_ll = gpd.GeoSeries([origin_pt_6670], crs=6670).to_crs(4326).iloc[0]

    # 1) snap the origin point to a graph node
    # 1) 출발지 Point -> 그래프 노드로 스냅
    try:
        u = ox.distance.nearest_nodes(G, origin_ll.x, origin_ll.y)
    except Exception as e:
        fails.append({"region_nm": region_nm, "type": "snap_src", "error": str(e)})
        continue

    # 2) every node reachable within the cutoff distance
    # 2) 컷오프 거리 이내에 도달 가능한 모든 노드
    try:
        dist_map = nx.single_source_dijkstra_path_length(G, source=u, cutoff=CUTOFF_M, weight="length")
    except Exception as e:
        fails.append({"region_nm": region_nm, "type": "dijkstra_cutoff", "error": str(e)})
        continue

    # 3) keep only the reachable nodes that are actually stops
    # 3) 도달 가능한 노드 중 실제 정류장인 것만 남김
    candidate_stop_nodes = [n for n in dist_map if n in stop_node_set]
    if not candidate_stop_nodes:
        fails.append({"region_nm": region_nm, "type": "no_stop_within_15min", "error": ""})
        continue

    # 4) rebuild the actual route (as a line) to each reachable stop node
    # 4) 도달 가능한 각 정류장 노드까지 실제 경로(선) 복원
    for v in candidate_stop_nodes:
        try:
            nodes = nx.shortest_path(G, u, v, weight="length")
            # Preserve actual OSM edge geometry (curves and parallel edges),
            # while using Dijkstra's edge-length sum as the authoritative distance.
            route_edges = ox.routing.route_to_gdf(G, nodes, weight="length")
            line_ll = route_edges.geometry.union_all()
            length_m = float(dist_map[v])
            duration_min = length_m / (WALK_SPEED_KMH * 1000 / 60)

            # 6) a node can carry several stops -- record all of them
            # 6) 한 노드에 정류장 여러 개가 붙을 수 있으므로 전부 기록
            for stop_i in node_to_stops.get(v, []):
                stop_row = gdf_stop_ll.loc[stop_i]
                routes.append({
                    "region_nm": region_nm,
                    "stop_type": stop_row.get("stop_type", ""),
                    "stop_nm": stop_row.get("stop_nm", ""),
                    "length_m": length_m,
                    "duration_min": duration_min,
                    "geometry": line_ll,
                })
        except Exception as e:
            fails.append({"region_nm": region_nm, "type": "route_build", "error": str(e)})

print(f"routes: {len(routes)} | fails: {len(fails)}")

##### 5) `ox.routing.route_to_gdf(G, nodes, weight="length")`
Rebuilds the route from the actual OSM edge geometries, preserving bends and
parallel-edge selection. The authoritative distance remains `dist_map[v]`,
the same edge-length sum used by Dijkstra.

실제 OSM 엣지 지오메트리로 경로를 복원해 곡선과 평행 엣지 선택을 보존한다.
거리에는 Dijkstra와 동일한 엣지 길이 합계인 `dist_map[v]`를 사용한다.

In [ ]:
route_columns = ["region_nm", "stop_type", "stop_nm", "length_m", "duration_min", "geometry"]
gdf_routes = gpd.GeoDataFrame(routes, columns=route_columns, geometry="geometry", crs=4326)
df_fails = pd.DataFrame(fails)

OUT_GPKG_ALL = os.path.join(OUT_DIR, "hiroshima_routes_all.gpkg")
OUT_FAIL_CSV = os.path.join(OUT_DIR, "hiroshima_routes_fail.csv")

df_fails.to_csv(OUT_FAIL_CSV, index=False, encoding="utf-8-sig")
if os.path.exists(OUT_GPKG_ALL):
    os.remove(OUT_GPKG_ALL)

In [ ]:
if len(gdf_routes) > 0:
    gdf_routes.to_file(OUT_GPKG_ALL, layer="routes_15min_all", driver="GPKG")

    gdf_routes_bus = gdf_routes[gdf_routes["stop_type"] == "bus"].copy()
    gdf_routes_station = gdf_routes[gdf_routes["stop_type"] == "station"].copy()

    if len(gdf_routes_bus) > 0:
        gdf_routes_bus.to_file(OUT_GPKG_ALL, layer="routes_15min_bus", driver="GPKG")
    if len(gdf_routes_station) > 0:
        gdf_routes_station.to_file(OUT_GPKG_ALL, layer="routes_15min_station", driver="GPKG")
else:
    print("[WARN] gdf_routes is empty. GPKG not created. Check the fails CSV.")

print("saved:")
print(" -", OUT_GPKG_ALL)
print(" -", OUT_FAIL_CSV)

In [ ]:
# summary: how close is the nearest stop, in real walking distance, from
# each part of these "0% covered" small areas?
# 요약: 이 "커버율 0%" 소지역들에서, 실제 도보 거리로 가장 가까운 정류장은
# 얼마나 가까운가?
summary = gdf_routes.groupby("region_nm")["length_m"].agg(["count", "min", "mean", "max"])
summary.columns = ["routes_found", "nearest_m", "mean_m", "farthest_m"]
summary

Both small areas have a real, walkable route to a bus stop or station
within 15 minutes -- so the "0% covered" finding from lecture 2 isn't
because transit is unreachable, but because the straight-line buffer
standard (300m bus / 500m rail) doesn't match how far people actually
have to walk here. The nearest real route is 500-600m in both areas --
close to, but past, the buffer thresholds. That's a meaningfully different
conclusion from "this area has no transit access at all."

두 소지역 모두 15분 이내에 실제로 걸어갈 수 있는 버스/역 경로가 있다 --
즉 2강의 "커버율 0%"는 교통이 아예 닿지 않아서가 아니라, 직선 버퍼
기준(버스 300m/철도 500m)이 실제 도보 거리와 안 맞아서 나온 결과다.
두 지역 다 가장 가까운 실제 경로가 500~600m로, 버퍼 기준을 살짝
넘어설 뿐이다. "이 지역엔 교통 접근성이 전혀 없다"와는 의미가 다른
결론이다.


In [ ]:
OUT_HTML = os.path.join(OUT_DIR, "hiroshima_lecture3_routes_15min.html")

bounds = gdf_admin.to_crs(4326).total_bounds
center_lat = (bounds[1] + bounds[3]) / 2
center_lon = (bounds[0] + bounds[2]) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles="cartodbpositron")

# (1) small-area boundary
# (1) 소지역 경계
folium.GeoJson(
    gdf_admin.to_crs(4326), name="small areas / 소지역",
    style_function=lambda x: {"fillOpacity": 0.05, "color": "#666666", "weight": 2},
    tooltip=folium.GeoJsonTooltip(fields=["region_nm"], aliases=["area / 지역"]),
).add_to(m)

# (2) uncovered area
# (2) 비커버 지역
folium.GeoJson(
    gdf_unc.to_crs(4326), name="uncovered (straight-line) / 비커버(직선기준)",
    style_function=lambda x: {"fillOpacity": 0.30, "color": "#cc0000", "weight": 1},
    tooltip=folium.GeoJsonTooltip(fields=["region_nm"], aliases=["area / 지역"]),
).add_to(m)

# (3) bus stop markers
# (3) 버스정류장 마커
fg_bus = folium.FeatureGroup(name="bus stops / 버스정류장", show=False)
for _, r in gdf_bus_sel.to_crs(4326).iterrows():
    folium.Marker(
        location=[r.geometry.y, r.geometry.x],
        icon=folium.Icon(color="blue", icon="bus", prefix="fa"),
        tooltip=f"bus / 버스 | {r.get('stop_nm','')}",
    ).add_to(fg_bus)
fg_bus.add_to(m)

# (4) origin grid points
# (4) 출발 격자점
fg_origin = folium.FeatureGroup(name="origin points / 출발 격자점", show=True)
for region_nm, pt_6670 in origins:
    pt_ll = gpd.GeoSeries([pt_6670], crs=6670).to_crs(4326).iloc[0]
    folium.CircleMarker(
        location=[pt_ll.y, pt_ll.x], radius=3,
        color="#7a3fe0", fill=True, fill_color="#a97fff", fill_opacity=0.9,
        tooltip=f"origin / 출발점 | {region_nm}",
    ).add_to(fg_origin)
fg_origin.add_to(m)

# (5) 15-minute walking routes, split by destination type
# (5) 15분 이내 도보 경로 (목적지 유형별로 분리)
fg_r_bus = folium.FeatureGroup(name="routes to bus / 경로(→버스)", show=True)
fg_r_station = folium.FeatureGroup(name="routes to station / 경로(→역)", show=False)

if len(gdf_routes) > 0:
    gdf_routes_ll = gdf_routes if str(gdf_routes.crs) == "EPSG:4326" else gdf_routes.to_crs(4326)
    for _, rr in gdf_routes_ll.iterrows():
        st = rr.get("stop_type", "")
        if st not in ("bus", "station"):
            continue
        color = "#0066ff" if st == "bus" else "#00aa55"
        tip = (
            f"{rr.get('region_nm','')} | "
            f"{'bus' if st=='bus' else 'station'} ({rr.get('stop_nm','')}) | "
            f"{rr.get('length_m',0):,.0f}m | approx. {rr.get('duration_min',0):.1f} min"
        )
        geojson = {"type": "Feature", "properties": {}, "geometry": mapping(rr.geometry)}
        folium.GeoJson(
            geojson,
            style_function=lambda x, c=color: {"color": c, "weight": 2, "opacity": 0.6},
            tooltip=tip,
        ).add_to(fg_r_bus if st == "bus" else fg_r_station)

fg_r_bus.add_to(m)
fg_r_station.add_to(m)

m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
folium.LayerControl(collapsed=False).add_to(m)
m.save(OUT_HTML)

print("saved:", OUT_HTML)
m

### Lecture 3 summary / 3강 요약

**Key takeaways / 핵심 요약**
- A straight-line buffer and a real walking route ask different questions -- "0% covered" (lecture 2) became "500-600m real walk" (this lecture), a materially different and more useful answer.
- Network analysis needs a directed graph mindset: nodes, edges, and a cutoff distance, not just "is it inside a circle."

**핵심 요약**
- 직선 버퍼와 실제 도보 경로는 서로 다른 질문이다. 2강의 "커버율 0퍼센트"가 이번 강의에서는 "실제 도보로 500~600미터"로 바뀌었다. 훨씬 더 유용하고 의미가 다른 답이다.
- 네트워크 분석에는 그래프 사고방식이 필요하다. "원 안에 있는가"가 아니라 노드, 엣지, 도달거리 상한이 기준이다.

**Real-world tip / 실무 팁**
`ox.distance.nearest_nodes()` silently snaps a coordinate to the nearest node even if that node is far outside your actual downloaded network extent -- it never raises an error for "this point is outside your graph." This project's own `app.py` hit exactly this: a buffer constant was 500m smaller than the distance stops could actually be selected from, so 36% of neighbour-included stops were silently snapping to the wrong edge node. If you buffer a network download by X metres, make sure every point you'll snap onto it is within X metres too.

**실무 팁**
`ox.distance.nearest_nodes()`는 좌표를 가장 가까운 노드에 스냅하는데, 그 노드가 실제로 다운로드한 네트워크 범위 훨씬 밖에 있어도 에러 없이 조용히 스냅해버린다. "이 지점은 그래프 밖입니다" 같은 에러를 절대 내지 않는다. 이 프로젝트의 `app.py`가 정확히 이 문제를 겪었다. 정류장을 고르는 거리 기준이 실제 네트워크 다운로드 범위보다 500미터 더 넓게 잡혀 있어서, 경계 밖 정류장의 36퍼센트가 조용히 엉뚱한 가장자리 노드로 스냅되고 있었다. 네트워크를 특정 거리만큼 버퍼로 받았다면, 거기에 스냅할 모든 점도 그 거리 이내인지 반드시 확인하자.

**Try it yourself / 직접 해보기**
Insert a new cell and try a shorter time budget (10 minutes) for one origin, without touching `routes` or `CUTOFF_M`.

새 셀을 추가해서 `routes`와 `CUTOFF_M`은 건드리지 않고, 한 출발점에 대해 더 짧은 시간 예산(10분)을 직접 시도해보자.

<details>
<summary>Show one way to do it / 정답 예시 보기</summary>

```python
my_cutoff_m = 4.8 * 1000 / 60 * 10  # 10 minutes instead of 15
region_nm, origin_pt = origins[0]
origin_ll = gpd.GeoSeries([origin_pt], crs=6670).to_crs(4326).iloc[0]
u = ox.distance.nearest_nodes(G, origin_ll.x, origin_ll.y)
reachable = nx.single_source_dijkstra_path_length(G, source=u, cutoff=my_cutoff_m, weight="length")
print(f"{len([n for n in reachable if n in stop_node_set])} stop(s) reachable within 10 min")
```

</details>

---
### Break (10 min) / 휴식 (10분)

Lecture 3 is done, one to go. Lecture 4 renders a 3D map through a
self-hosted basemap server -- **start it now, during the break**, so
it's ready when we get there:

```
tools\pmtiles.exe serve data/hiroshima --port 8891 --cors "*" --public-url "http://localhost:8891/"
python -m http.server 8001   (from the repo root, in a second terminal)
```

If you skip this and have internet, it falls back to a CDN basemap
automatically -- but the self-hosted version is what makes it work
without conference wifi.

3강이 종료되었다, 한 강 남았다. 4강은 자체 호스팅한 배경지도 서버로
3D 지도를 렌더링한다 -- **휴식 시간 중 지금 미리 켜두면** 4강 시작할 때
바로 쓸 수 있다:

```
tools\pmtiles.exe serve data/hiroshima --port 8891 --cors "*" --public-url "http://localhost:8891/"
python -m http.server 8001   (저장소 루트에서, 다른 터미널에서)
```

안 켜도 인터넷이 있으면 자동으로 CDN 배경지도로 대체되지만,
컨퍼런스 wifi 없이도 돌아가게 하는 건 자체 호스팅 버전이다.
---

### [Lecture 4] Where would a new stop help the most?
### [4강] 어디에 놓아야 가장 효과적인가

Lecture 3 confirmed, via real walking routes, how far residents of the two
most vulnerable small areas have to walk today. Lecture 4 asks the natural
follow-up: **for a given search radius, how much population is actually
inside it, once you account for cells that only partly overlap** -- and
renders the answer as an interactive 3D map, extruding each small area by
its population.

3강은 실제 도보 경로로 취약 소지역 두 곳의 오늘의 접근 거리를 확인했다.
4강은 자연스러운 다음 질문을 다룬다: **주어진 탐색 반경 안에 실제로 얼마나
많은 인구가 있는가 -- 반경에 일부만 걸친 구역까지 정확히 반영해서** --
그리고 그 답을 각 소지역을 인구에 비례해 압출한 인터랙티브 3D 지도로
그린다.

This lecture also switches rendering tools: lecture 3 used Folium (Leaflet-
based, strong at 2D interactivity -- pan, zoom, layer toggles, tooltips).
This lecture uses **pydeck**, a deck.gl binding
(GPU-accelerated 3D visualization -- height, colour, and camera angle to
show *how much* and *how concentrated*, not just *where*). Restored here
as literal, runnable pydeck code rather than the lonboard port used
earlier in this project -- see the cell below for what that trades off.

이번 강의는 렌더링 도구도 바꾼다: 3강은 Folium(Leaflet 기반, 확대·이동·
레이어 토글·툴팁 같은 2D 인터랙션에 강함)을 썼다. 이번 강의는 deck.gl
바인딩인 **pydeck**을 쓴다 (GPU 가속 3D 시각화 -- 높이·색상·
카메라 각도로 "어디에 있는가"뿐 아니라 "얼마나 많고 얼마나 밀집했는가"를
보여준다). 이 프로젝트에서 한때 이 자리를 대체했던 lonboard 포팅 대신,
실제로 실행되는 pydeck 코드 그대로 복원했다 -- 그 트레이드오프는 바로
아래 셀 참고.

*A note on the population layer: a fine 100m population grid would be
the ideal population surface here, distinct from lecture 1's
registered-population-by-administrative-area table. Hiroshima's e-Stat
source doesn't offer an equivalent fine grid in this project's data
pipeline, so this lecture reuses lecture 1's small-area polygons
(`hiroshima_city_admin.gpkg`) as the population surface instead. The
technique below -- fractional area-weighting for partial overlap -- works
identically regardless of whether the underlying shapes are a uniform
grid or irregular administrative polygons; only the visual grain differs.*

*인구 레이어 참고: 세밀한 100m 인구격자가 여기서는 이상적인 인구 표면이겠지만,
1강의 등록인구(행정구역 단위) 테이블과는 별개다. 히로시마의 e-Stat
출처는 이 프로젝트의 데이터 파이프라인 안에서 그에 대응하는 세밀한 격자를
제공하지 않아서, 이번 강의는 1강의 소지역 폴리곤(`hiroshima_city_admin.gpkg`)을
인구 표면으로 대신 재사용한다. 아래에서 쓰는 기법(부분 중첩에 대한 면적가중)은
바탕 도형이 균일한 격자든 불규칙한 행정구역 폴리곤이든 동일하게 작동한다 --
시각적 결 차이만 있을 뿐이다.*

This lecture's elevation formula was tuned for a 100m population grid,
and it was the fastest line in this whole notebook to write -- also the
easiest place for that speed to backfire: unchanged, the same formula
turned Hiroshima's larger chōme polygons into 3D columns 31km tall. The
fix took one look at the rendered map and a percentile-based rescale --
but only because someone looked, instead of assuming a formula tuned for
one grid size would carry over to another unchanged.

이번 강의의 고도 계산 공식은 100m 인구격자에 맞춰져 있었고, 이 노트북
전체에서 가장 빨리 쓸 수 있었던 코드 한 줄이었다 -- 동시에 그 속도가
가장 쉽게 역효과를 내는 지점이기도 했다: 공식을 그대로 썼더니 히로시마의
더 큰 소지역 폴리곤이 31km 높이의 3D 기둥으로 치솟았다. 수정은 렌더링된
지도를 한 번 보고 백분위수 기반으로 재조정하는 것으로 끝났다 -- 다만
그건 하나의 격자 크기에 맞춘 공식이 다른 크기에도 그대로 넘어올 거라
가정하는 대신 실제로 봤을 때만 가능했다.


In [ ]:
# Not run here: pydeck is already installed in this notebook's uv
# environment (see cell 2's note); %pip install would fail loudly for no
# reason in this venv.
# 여기서는 실행하지 않음: pydeck은 이 노트북의 uv 환경에 이미
# 설치되어 있다(셀 2의 설명 참고). 이 venv에서는 %pip install이 굳이
# 큰 소리로 실패한다.
# %pip install pydeck


#### Folium vs. pydeck
#### Folium과 pydeck의 차이

Both can render web maps, but their strengths differ. Folium (Leaflet-based)
suits 2D interactive maps -- boundaries, points, polygons, lines, with pan/
zoom/layer-toggle/tooltip built in; it's easy to drop into a report or web
page. pydeck (deck.gl, GPU-accelerated) is built for large-scale 3D
visualization -- extrusion height, colour, and camera angle make spatial
*intensity* differences legible, not just spatial *location*.

So: Folium for understanding spatial structure and coverage (lecture 3),
pydeck for comparing population concentration and scenario differences
(this lecture) -- each tool used where its strength actually applies.

둘 다 웹 지도를 그릴 수 있지만 강점이 다르다. Folium(Leaflet 기반)은
2D 인터랙티브 지도에 적합하다 -- 경계·점·폴리곤·라인을 확대·이동·레이어
토글·툴팁과 함께 표시하기 좋고, 보고서·웹 문서에 넣기 쉽다. pydeck
(deck.gl, GPU 가속)은 대규모 3D 시각화에 특화되어 있다 -- 높이(압출)·
색상·카메라 각도로 공간의 "위치"뿐 아니라 "강도" 차이를 드러낸다.

그래서 공간 구조와 커버리지를 이해하는 단계(3강)에는 Folium을, 인구 집중도와
시나리오 차이를 비교하는 단계(이번 강의)에는 pydeck을 쓴다 -- 각 도구를
실제로 강점이 발휘되는 곳에 쓰는 접근이다.

**A real tradeoff, not free**: pydeck has no native GeoPandas integration
(unlike lonboard's `.from_geopandas()`) -- a `MultiPolygon` has to be
exploded into single `Polygon` rows by hand before pydeck can draw it, and
its default Mapbox-branded styles need a **free Mapbox access token**
(`MAPBOX_TOKEN`, see the basemap cell below) to actually reach Mapbox's
tile servers over the internet. This is the literal reason the earlier
lonboard version of this lecture existed in the first place -- restored
here anyway, deliberately, so this lecture also teaches the tool most
GIS Python users will actually meet first.

**실제로 존재하는 트레이드오프**: pydeck은 lonboard의 `.from_geopandas()`
같은 GeoPandas 네이티브 통합이 없다 -- `MultiPolygon`을 pydeck이 그리려면
직접 낱개 `Polygon` 행으로 분해해야 한다. 그리고 기본 Mapbox 스타일을 쓰려면
**무료 Mapbox 액세스 토큰**(`MAPBOX_TOKEN`, 아래 배경지도 셀 참고)이 있어야
인터넷을 통해 Mapbox 타일 서버에 접근할 수 있다. 이게 바로 이 강의가 한때
lonboard로 바뀌었던 이유다 -- 그럼에도 GIS Python 사용자가 실무에서 가장
먼저 마주칠 가능성이 높은 도구이기도 해서, 의도적으로 다시 여기 pydeck으로
되돌렸다.

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.ops import unary_union

# pydeck: a deck.gl binding. Its default map_provider
# is "carto" (free, no token) -- a Mapbox-branded basemap is opt-in via
# map_provider="mapbox" plus a real access token, set up in the basemap
# cell below.
# pydeck: deck.gl 바인딩. 기본 map_provider는 "carto"
# (무료, 토큰 불필요)이고, Mapbox 브랜드 배경지도는 map_provider="mapbox"와
# 실제 액세스 토큰을 함께 지정해야 쓸 수 있다 (아래 배경지도 셀에서 설정).
import pydeck as pdk

from IPython.display import display


#### 2. Population surface / 2. 인구 표면

In [ ]:
# reuse lecture 1's small-area polygons + population, per the note above
# 위 설명대로 1강의 소지역 폴리곤 + 인구를 재사용
GRID_ID_COL = "region_id"
GRID_POP_COL = "population"

gdf_grid = gpd.read_file(os.path.join(DATA_DIR, "hiroshima_city_admin.gpkg"))
gdf_grid = gdf_grid.rename(columns={"KEY_CODE": "region_id", "pop": "population"})
if gdf_grid.crs is None:
    raise ValueError("grid source has no CRS")
gdf_grid = gdf_grid.to_crs(6670)

# Same region_id duplication as lecture 1 (see cell 15's note) -- this reads
# the raw file independently rather than reusing lecture 1's already-
# dissolved gdf_admin, so it needs the same fix applied here too.
# 1강과 같은 region_id 중복 문제(cell 15의 설명 참고) -- 여기서는 1강에서
# 이미 dissolve한 gdf_admin을 재사용하지 않고 원본 파일을 독립적으로 다시
# 읽으므로, 여기서도 같은 수정을 적용해야 한다.
gdf_grid = gdf_grid.dissolve(
    by="region_id",
    aggfunc={"region_nm": "first", "ward_nm": "first", "population": "sum"},
).reset_index()
assert gdf_grid["region_id"].is_unique, "region_id must be unique before any downstream merge"

In [ ]:
gdf_grid["pop"] = pd.to_numeric(gdf_grid[GRID_POP_COL], errors="coerce").fillna(0).astype(float)

In [ ]:
UNCOVERED_GPKG = OUT_DIR / "hiroshima_uncovered_city.gpkg"
if not UNCOVERED_GPKG.exists():
    raise FileNotFoundError(
        "Run Lecture 2 first to create outputs/hiroshima_uncovered_city.gpkg"
    )
gdf_unc = gpd.read_file(UNCOVERED_GPKG, layer="uncovered_city")
if gdf_unc.crs is None:
    raise ValueError("uncovered layer has no CRS")
gdf_unc = gdf_unc.to_crs(6670)

In [ ]:
# reload bus/station data city-wide (not the lecture-3 within-demo-area subset)
# 시내 전역 버스/역 데이터를 다시 로드 (3강의 데모지역-내부 한정 부분집합이 아님)
gdf_bus_all = gpd.read_file(os.path.join(DATA_DIR, "hiroshima_city_bus_stops.gpkg")).to_crs(6670)
gdf_station_all = gpd.read_file(os.path.join(DATA_DIR, "hiroshima_city_stations.gpkg")).to_crs(6670)
gdf_station_all["geometry"] = gdf_station_all.geometry.centroid  # platform line -> point, as before
                                                                    # 플랫폼 선분 -> 점, 이전과 동일

gdf_transit = gpd.GeoDataFrame(
    pd.concat([gdf_bus_all[["geometry"]], gdf_station_all[["geometry"]]], ignore_index=True),
    geometry="geometry",
    crs=6670,
)

In [ ]:
RADIUS_M = 1250.0  # search radius around the selected candidate

# Compute CITY-WIDE uncovered fractions before selecting the local radius.
unc_union = gdf_unc.geometry.union_all()
valid_area = gdf_grid.geometry.area > 0
gdf_grid["unc_frac"] = 0.0
gdf_grid.loc[valid_area, "unc_frac"] = (
    gdf_grid.loc[valid_area].geometry.intersection(unc_union).area
    / gdf_grid.loc[valid_area].geometry.area
).clip(0, 1)
gdf_grid["is_uncovered"] = gdf_grid["unc_frac"] > 0

# Nearest-stop distance is an indicator, not a prefilter for the search circle.
gdf_grid_cent = gpd.GeoDataFrame(
    gdf_grid[[GRID_ID_COL, "pop"]].copy(),
    geometry=gdf_grid.geometry.representative_point(),
    crs=6670,
)
gdf_nn = gpd.sjoin_nearest(
    gdf_grid_cent, gdf_transit[["geometry"]], how="left", distance_col="dist_m"
)
nearest_dist = gdf_nn.groupby(GRID_ID_COL)["dist_m"].min()
gdf_grid["nearest_stop_m"] = gdf_grid[GRID_ID_COL].map(nearest_dist)
print(f"small areas prepared: {len(gdf_grid):,}")

The nearest-stop distance above is retained as a diagnostic indicator.
It is deliberately **not** used to discard polygons before the search circle is
built: a large polygon can overlap the circle even when its representative point
is farther away.

위 최근접 정류장 거리는 진단 지표로만 유지한다. 큰 폴리곤은 대표점이 멀어도
탐색 원과 겹칠 수 있으므로, 탐색 원을 만들기 전에 후보를 제거하는 필터로
사용하지 않는다.

In [ ]:
# RADIUS_M is defined once in the preparation cell above.

In [ ]:
# No centroid-based candidate prefilter.

In [ ]:
# Every small area remains eligible as a centre candidate.
gdf_cand = gdf_grid.copy()
print(f"candidate small areas: {len(gdf_cand):,}")

In [ ]:
# City-wide unc_union was computed above.

In [ ]:
# Uncovered fractions were calculated city-wide in the preparation cell.
assert {"is_uncovered", "unc_frac"}.issubset(gdf_grid.columns)

In [ ]:
# All grid IDs remain available.

In [ ]:
# the more vulnerable of the two small areas from lecture 2 -- change this
# value to re-run the analysis centred on a different candidate
# 2강에서 나온 두 소지역 중 더 취약한 쪽 -- 이 값을 바꾸면 다른 후보를
# 중심으로 재실행할 수 있다
sel_name = "海老園四丁目"

matches = gdf_cand[gdf_cand["region_nm"] == sel_name]
if len(matches) == 0:
    auto_row = gdf_cand.sort_values("pop", ascending=False).iloc[0]
    print(f"[WARN] sel_name={sel_name!r} not among candidates -> auto-selecting {auto_row['region_nm']!r}")
    sel_gid = str(auto_row[GRID_ID_COL])
else:
    sel_gid = str(matches.iloc[0][GRID_ID_COL])

In [ ]:
sel_poly = gdf_cand.loc[gdf_cand[GRID_ID_COL].astype(str) == sel_gid, "geometry"].iloc[0]

In [ ]:
sel_center = sel_poly.centroid

In [ ]:
circle_6670 = sel_center.buffer(RADIUS_M)

In [ ]:
in_circle = gdf_grid.geometry.intersects(circle_6670)

In [ ]:
gdf_in = gdf_grid.loc[
    in_circle,
    [GRID_ID_COL, "region_nm", "pop", "nearest_stop_m", "is_uncovered", "unc_frac", "geometry"],
].copy()

# A cell only partly inside the radius should contribute only that part.
# 반경에 일부만 걸친 구역은 그 비율만큼만 기여해야 한다.
gdf_in["in_frac"] = (
    gdf_in.geometry.intersection(circle_6670).area / gdf_in.geometry.area
).clip(0, 1)

# Effective population = population x share of the cell inside the radius.
# 유효 인구 = 인구 x 반경 안에 들어온 구역 비율.
gdf_in["pop_eff"] = gdf_in["pop"] * gdf_in["in_frac"]

assert len(gdf_in) > 0, "Search circle contains no small-area polygons"
print(f"small areas intersecting the search circle: {len(gdf_in):,}")

In [ ]:
# Every figure below is area-weighted: a partial overlap contributes partially.
# 아래 모든 수치는 면적가중이다. 부분 중첩은 부분만 기여한다.

total_pop = float(gdf_in["pop_eff"].sum())
unc_pop   = float((gdf_in["pop_eff"] * gdf_in["unc_frac"]).sum())
cov_pop   = total_pop - unc_pop
unc_rate  = (unc_pop / total_pop) if total_pop > 0 else 0.0

# The naive (unweighted, whole-cell) figures, kept so the difference stays visible.
# 원본(비가중, 전체 구역 계상) 수치. 차이를 계속 보이게 하려고 함께 보존한다.
total_pop_raw = float(gdf_in["pop"].sum())
unc_pop_raw   = float(gdf_in.loc[gdf_in["is_uncovered"] == True, "pop"].sum())

print("\n======== KPI (area-weighted) ========")
print("selected small area:", sel_name if len(matches) else "auto")
print("small areas in radius:", int(len(gdf_in)))
print(f"total_pop:      {total_pop:,.0f}   (unweighted {total_pop_raw:,.0f})")
print(f"uncovered_pop:  {unc_pop:,.0f}   (unweighted {unc_pop_raw:,.0f})")
print(f"covered_pop:    {cov_pop:,.0f}")
print(f"uncovered_rate: {unc_rate*100:.2f}%")
if total_pop > 0:
    print(f"overstatement if unweighted -- total: {total_pop_raw-total_pop:+,.0f} ({100*(total_pop_raw-total_pop)/total_pop:+.1f}%)")
if unc_pop > 0:
    print(f"overstatement if unweighted -- uncovered: {unc_pop_raw-unc_pop:+,.0f} ({100*(unc_pop_raw-unc_pop)/unc_pop:+.1f}%)")
print("======================================\n")

Skipping area-weighting here comes at a real cost: **+39% for total
population and +350% for uncovered population**, overstated. That's
not a coincidence: it's a direct consequence of substituting chōme polygons
for a fine grid, exactly the tradeoff flagged in the note at the top of
this lecture. A 1,250m search circle only intersects a handful of large
chōme polygons, so each partially-overlapping one carries far more weight
than a partially-overlapping 100m grid cell would. The lesson isn't
"chōme polygons are wrong" -- it's that **the coarser your spatial units,
the more area-weighting matters**, not less.

면적가중을 건너뛰면 실제로 대가가 따른다: **전체 인구 +39%, 비커버
인구 +350%**가 과대계상된다. 우연이 아니다: 이 강의 맨 앞에서 언급한
대로, 세밀한 격자 대신 소지역 폴리곤을 쓴 것의
직접적인 결과다. 1,250m 반경 원은 크기가 큰 소지역 폴리곤 몇 개만
교차하므로, 일부만 겹치는 폴리곤 하나하나가 100m 격자 셀 하나가 일부만
겹칠 때보다 훨씬 큰 비중을 차지한다. 여기서 얻을 교훈은 "소지역 폴리곤을
쓰면 안 된다"가 아니라, **공간 단위가 성길수록 면적가중의 중요성은
줄어드는 게 아니라 오히려 커진다**는 것이다.


#### Mapbox GL and pydeck's basemap providers
#### Mapbox GL과 pydeck의 배경지도 제공자

pydeck renders through **deck.gl**, and its basemap can come from either
of two providers:

- `map_provider="carto"` (pydeck's default): free, no API key, Carto's
  hosted vector tiles.
- `map_provider="mapbox"`: Mapbox's own hosted styles (`mapbox://styles/...`)
  -- richer built-in styles, but requires a **Mapbox access token** and a
  live connection to Mapbox's servers.

pydeck은 **deck.gl**로 렌더링하며, 배경지도는 두 제공자 중 하나에서 온다:

- `map_provider="carto"` (pydeck 기본값): 무료, API 키 불필요, Carto가
  호스팅하는 벡터 타일.
- `map_provider="mapbox"`: Mapbox가 직접 호스팅하는 스타일
  (`mapbox://styles/...`) -- 기본 제공 스타일이 더 다양하지만, **Mapbox
  액세스 토큰**과 Mapbox 서버로의 실시간 연결이 필요하다.

- pydeck: https://deckgl.readthedocs.io/
- Mapbox access tokens: https://account.mapbox.com/access-tokens/

In [ ]:
# --- Basemap style ---------------------------------------------------------
# This lecture's map defaults to Mapbox's hosted style when a token is
# available (MAPBOX_TOKEN environment variable). Without a token it falls
# back to pydeck's built-in Carto basemap -- but that fallback needs an
# EXPLICIT style alias ("light"/"dark"/"road"); map_style=None renders NO
# basemap at all in pydeck 0.9.3's to_html() export. Confirmed directly
# (zero Carto tile/style requests fire, no exception, just an empty canvas
# under the 3D population blocks) -- the exact silent-failure shape this
# whole project is about, this time in the verification tooling itself
# rather than the analysis. "dark" is used here to match the Mapbox
# dark-v11 style used in the token branch above.
#
# Get a free token at https://account.mapbox.com/access-tokens/ and set it
# BEFORE starting Jupyter (never hardcode a real token into this notebook --
# this repo has already had exposed tokens revoked once, see CLAUDE.md):
#   macOS/Linux:  export MAPBOX_TOKEN=pk.your_token_here
#   PowerShell:   $env:MAPBOX_TOKEN = "pk.your_token_here"
#
# 이 강의의 지도는 토큰이 있으면(MAPBOX_TOKEN 환경변수) Mapbox 호스팅
# 스타일을 기본으로 쓴다. 토큰이 없으면 pydeck 내장 Carto 배경지도로
# 대체하는데, 이 대체 경로는 명시적인 스타일 별칭("light"/"dark"/"road")이
# 있어야 동작한다 -- map_style=None을 주면 pydeck 0.9.3의 to_html() 결과물에는
# 배경지도가 아예 렌더링되지 않는다. 직접 확인함(Carto 타일·스타일 요청이
# 0건, 에러도 없이 3D 인구블록 아래가 그냥 빈 캔버스가 된다) -- 이 프로젝트가
# 경계하는 바로 그 조용한 실패 유형이 이번엔 분석이 아니라 검증 도구 자체에서
# 재현된 것. 위 토큰 분기의 Mapbox dark-v11과 시각적으로 맞추려고 "dark"를 썼다.
#   macOS/Linux:  export MAPBOX_TOKEN=pk.your_token_here
#   PowerShell:   $env:MAPBOX_TOKEN = "pk.your_token_here"

MAPBOX_TOKEN = os.environ.get("MAPBOX_TOKEN")

if MAPBOX_TOKEN:
    MAP_PROVIDER = "mapbox"
    MAP_STYLE = "mapbox://styles/mapbox/dark-v11"
    print("MAPBOX_TOKEN found -- using Mapbox's hosted style (needs live internet).")
else:
    MAP_PROVIDER = "carto"
    MAP_STYLE = "dark"  # explicit alias required -- None silently renders no basemap
    print("MAPBOX_TOKEN not set -- using pydeck's free Carto basemap (style='dark').")
    print("Set MAPBOX_TOKEN and re-run this cell for the Mapbox-branded version.")


In [ ]:
gdf_ll = gdf_in.to_crs(4326).copy()

In [ ]:
# NOTE on the elevation formula: this lecture's elevation formula was
# originally fixed as pop**1.80 * 0.02 -- a scale tuned for a 100m grid, where each cell holds a
# few hundred people at most. This lecture's population unit is a chome
# polygon (thousands of people each), so that fixed scale produced 30km-tall
# columns off the top of the screen -- a real bug caught by actually looking
# at the rendered map, not just checking that the cell ran without error.
# Normalizing by the 99.5th-percentile population before exponentiating
# keeps the visual height sane regardless of whether the input is a fine
# grid or coarse administrative polygons.
#
# Also uses pop_eff (area-weighted, from the KPI cell above), not the raw
# pop column: the markdown intro to this lecture explicitly promises a map
# that accounts for "cells that only partly overlap" the search radius --
# using raw pop here would extrude every partially-overlapping small area
# to its FULL population height, silently contradicting the exact
# area-weighting lesson this lecture's own KPI printout demonstrates
# (the +39%/+350% overstatement box above). Caught by reviewing this
# lecture's code against its own stated claim, not by an execution error.
# 표고(높이) 공식 참고: 이 강의의 고도 공식은 원래 pop**1.80 * 0.02로 고정돼 있었다 --
# 셀 하나가 최대 수백 명인 100m 격자에 맞춘 값이다. 이번 강의의 인구 단위는
# 소지역 폴리곤(각각 수천 명)이라 그 고정 스케일을 그대로 쓰면 화면 밖으로
# 솟는 30km짜리 기둥이 나온다 -- 셀이 에러 없이 실행됐는지만 확인하지 않고
# 실제 렌더링된 지도를 직접 봐서 잡은 실제 버그다. 지수 계산 전에 인구를
# 99.5백분위수로 정규화하면, 입력이 세밀한 격자든 성긴 행정구역 폴리곤이든
# 시각적으로 타당한 높이가 유지된다.
#
# raw pop이 아니라 (위 KPI 셀에서 만든) 면적가중 pop_eff를 쓴다: 이 강의
# 도입부 markdown이 "반경에 일부만 걸친 구역까지 정확히 반영해서"라고 명시
# 약속했는데, raw pop을 쓰면 반경에 일부만 걸친 소지역도 전체 인구만큼
# 압출되어, 바로 위 KPI 출력(+39%/+350% 과대계상)이 보여주는 면적가중
# 교훈을 지도 자체가 조용히 어기게 된다. 실행 에러가 아니라 이 강의 코드를
# 자신이 한 말과 대조해서 잡은 문제다.
pop = gdf_ll["pop_eff"].clip(lower=0).astype(float)
cap_val = float(pop.quantile(0.995)) if pop.quantile(0.995) > 0 else float(pop.max())
pop_capped = np.minimum(pop, cap_val)
pop_ratio = pop_capped / cap_val if cap_val > 0 else pop_capped * 0.0
MAX_ELEV_M = 900.0  # visual extrusion cap, metres / 시각적 압출 높이 상한(m)
gdf_ll["elev"] = (np.power(pop_ratio, 1.80) * MAX_ELEV_M).astype(float)


In [ ]:
p_red = float(np.quantile(pop_capped, 0.95))
p80   = float(np.quantile(pop_capped, 0.80))
p50   = float(np.quantile(pop_capped, 0.50))
p20   = float(np.quantile(pop_capped, 0.20))

#### pydeck core concepts to know
#### 꼭 알아야 할 pydeck 핵심 함수·개념 정리

##### 1. `pdk.Layer` (e.g. `"PolygonLayer"`)
pydeck's core visualization unit, named by string (`"PolygonLayer"`,
`"ScatterplotLayer"`, ...) rather than imported as a class. Unlike
lonboard, it has **no native GeoPandas support** -- `data=` takes a plain
`pandas.DataFrame` whose geometry column is already a list of `[lon, lat]`
coordinate pairs, not a `GeoDataFrame`.

pydeck의 핵심 시각화 단위. lonboard와 달리 클래스를 import하는 대신
문자열로 지정한다(`"PolygonLayer"`, `"ScatterplotLayer"` 등). GeoPandas
네이티브 지원이 **없어서**, `data=`에는 geometry 컬럼이 이미 `[lon, lat]`
좌표쌍 리스트로 바뀐 일반 `pandas.DataFrame`을 넘겨야 한다.

##### 2. Accessors take a column-name string
`get_elevation="elev"`, `get_fill_color="fill_color"` -- pydeck looks up
these strings as columns on the DataFrame at render time. (This is the
opposite of lonboard, which wants the actual array.)

`get_elevation="elev"`, `get_fill_color="fill_color"`처럼 컬럼 이름
문자열을 넘기면 렌더링 시점에 pydeck이 DataFrame에서 그 컬럼을 찾는다.
(실제 배열을 넘기는 lonboard와는 반대 방향이다.)

##### 3. `pdk.ViewState`: the camera's position and angle
`latitude`/`longitude` (map centre), `zoom`, `pitch` (tilt -- the core of
the 3D effect), `bearing` (rotation). Same concept as lonboard's
`MapViewState`, different class name.

지도 중심 좌표(`latitude`/`longitude`), `zoom`, `pitch`(기울기, 3D 효과의
핵심), `bearing`(회전 각도). lonboard의 `MapViewState`와 개념은 같고
클래스 이름만 다르다.

##### 4. `pdk.Deck`: bundles every layer and the view into one map object
`map_provider`/`map_style`/`api_keys` control the basemap (see the cell
above); leaving it as a cell's last line renders it as a live widget in
Jupyter.

`map_provider`/`map_style`/`api_keys`로 배경지도를 제어한다(위 셀 참고).
셀의 마지막 줄에 두면 Jupyter에서 바로 위젯으로 렌더링된다.

##### 5. `Deck.to_html()`: saves the map as a standalone HTML file
Runs independent of Jupyter/VS Code/Colab -- ready to use directly for a
report or a live demo. **Unlike the self-hosted-PMTiles version this
lecture used with lonboard, a Mapbox-provider export still needs internet
+ a valid token at viewing time**, not just at build time -- worth knowing
before relying on it at a venue with unreliable wifi.

Jupyter/VS Code/Colab 환경과 무관하게 실행 가능 -- 보고서나 발표 데모에
바로 활용할 수 있다. **이 강의가 lonboard로 쓰던 자체 호스팅 PMTiles
버전과 달리, Mapbox provider로 내보낸 결과물은 빌드 시점뿐 아니라 나중에
볼 때도 인터넷과 유효한 토큰이 필요하다** -- wifi가 불안정한 발표장에서는
이 점을 미리 감안해야 한다.

In [ ]:
# colour by population size (pale yellow -> orange -> red for the top 5%)
# 인구 규모에 따른 색상 지정 (연노랑 → 주황 → 상위 5%만 빨강)
fill_colors = []
for p, is_unc in zip(pop_capped, gdf_ll["is_uncovered"]):
    if p <= p20:
        c = [255, 248, 210, 120]
    elif p <= p50:
        c = [255, 232, 160, 140]
    elif p <= p80:
        c = [255, 200, 120, 165]
    elif p <= p_red:
        c = [245, 150, 95, 190]
    else:
        c = [210, 70, 60, 220]

    # uncovered areas render more opaque, covered areas more transparent
    # 비커버 지역은 더 진하게, 커버 지역은 투명도를 낮춘다
    if bool(is_unc):
        c[3] = min(255, c[3] + 25)
    else:
        c[3] = max(60, c[3] - 35)

    fill_colors.append(c)

gdf_ll["fill_color"] = fill_colors

#### Visualization CRS convention (EPSG:4326)
#### 시각화 좌표계 사용 원칙 (EPSG:4326)

**pydeck (and the deck.gl stack underneath it) must use EPSG:4326 (WGS84,
lat/lon) at the visualization stage.** WebGL-based map renderers
universally assume latitude/longitude, regardless of whether the basemap
provider is Mapbox or Carto.

**pydeck(및 그 기반인 deck.gl)은 시각화 단계에서 반드시 EPSG:4326(WGS84,
경·위도 좌표계)을 써야 한다.** WebGL 기반 지도 렌더러는 배경지도 제공자가
Mapbox든 Carto든 위도·경도 좌표계를 전제로 지도를 그린다.

##### Recommended flow
##### 권장 좌표계 흐름
1. **Analysis stage (GeoPandas / Shapely)**: a metric projected CRS (e.g. EPSG:6670) for distance/buffer/nearest-join operations
2. **Visualization stage (pydeck)**: `.to_crs(4326)`, then convert each polygon's exterior ring into a plain `[[lon, lat], ...]` list column before handing it to `pdk.Layer`

1. **분석 단계 (GeoPandas · Shapely)**: 거리·버퍼·최근접조인 연산을 위한 미터 단위 투영좌표계 (예: EPSG:6670)
2. **시각화 단계 (pydeck)**: `.to_crs(4326)`으로 변환한 뒤, 각 폴리곤의 외곽 좌표를 `[[lon, lat], ...]` 형태의 일반 리스트 컬럼으로 바꿔서 `pdk.Layer`에 전달

In [ ]:
# ring showing the search radius
# 탐색 반경을 표시할 링(원형 라인) 생성
circle_ll = gpd.GeoSeries([circle_6670], crs=6670).to_crs(4326).iloc[0]

In [ ]:
# 3D population-block layer.
# pydeck has no from_geopandas() -- unlike lonboard, it needs a plain
# DataFrame whose geometry is already [[lon, lat], ...] coordinate lists,
# and it can't draw a MultiPolygon as one row, so multi-part small areas
# are exploded into one row per constituent Polygon first.
# 3D 인구블록 레이어.
# pydeck은 from_geopandas()가 없다 -- lonboard와 달리 geometry가 이미
# [[lon, lat], ...] 좌표 리스트인 일반 DataFrame이 필요하고, MultiPolygon을
# 한 행으로 그릴 수 없어서, 여러 조각으로 된 소지역은 먼저 조각(Polygon)
# 하나당 한 행으로 explode한다.
_block_rows = []
for _, r in gdf_ll.iterrows():
    geom = r.geometry
    parts = list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]
    for part in parts:
        _block_rows.append({
            "polygon": [[x, y] for x, y in part.exterior.coords],
            "elev": float(r["elev"]),
            "fill_color": r["fill_color"],
            "region_nm": r["region_nm"],
        })
df_blocks = pd.DataFrame(_block_rows)

layer_blocks = pdk.Layer(
    "PolygonLayer",
    data=df_blocks,
    get_polygon="polygon",
    extruded=True,
    filled=True,
    get_elevation="elev",
    elevation_scale=1,
    get_fill_color="fill_color",
    pickable=True,
    auto_highlight=True,
)


In [ ]:
# ring layer showing the search radius -- same explode-to-plain-coordinates
# step as the block layer above, just for a single ring polygon
# 탐색 반경을 표시하는 원형 링 레이어 -- 위 블록 레이어와 동일하게 좌표를
# 일반 리스트로 바꾸는 과정을 원(circle) 폴리곤 하나에 적용
_ring_coords = [[x, y] for x, y in circle_ll.exterior.coords]
df_ring = pd.DataFrame([{"polygon": _ring_coords}])

layer_circle = pdk.Layer(
    "PolygonLayer",
    data=df_ring,
    get_polygon="polygon",
    filled=False,
    stroked=True,
    get_line_color=[255, 255, 255, 220],
    get_line_width=40,
)


#### Reading the 3D map / 3D 지도 읽는 법

- **Colour** -- pale yellow to red across five population quantiles (the
  bottom 20% palest, the top 5% reddest); each small area's opacity also
  increases if it falls in the uncovered set from lecture 2, so uncovered
  *and* populous areas stand out most.
- **Height** -- proportional to `pop_eff` (the area-weighted population
  actually inside the search circle, not raw population), capped at 900m
  purely for visual sanity -- see the note above on why this needed
  rescaling for Hiroshima's larger polygons.
- **White ring** -- the 1,250m search radius used for the KPI above.

- **색상** -- 인구 5분위에 따라 연노랑에서 빨강까지(하위 20%가 가장
  옅고, 상위 5%가 가장 진한 빨강). 2강의 비커버 지역에 해당하면 불투명도도
  높아져서, 비커버이면서 인구도 많은 지역이 가장 눈에 띈다.
- **높이** -- `pop_eff`(원 안에 실제로 들어온 면적가중 인구, raw 인구가
  아님)에 비례하며, 순전히 시각적 이유로 900m에서 상한을 둔다 -- 히로시마의
  더 큰 폴리곤에서 왜 재조정이 필요했는지는 위 참고 설명 참고.
- **흰색 원** -- 위 KPI에서 쓴 1,250m 탐색 반경.


In [ ]:
# camera centred on the selected small area
# 선택한 소지역 중심을 기준으로 카메라 설정
sel_center_ll = gpd.GeoSeries([sel_center], crs=6670).to_crs(4326).iloc[0]

view_state = pdk.ViewState(
    latitude=float(sel_center_ll.y),
    longitude=float(sel_center_ll.x),
    zoom=14.0,
    pitch=60,
    bearing=20,
)

# api_keys is only meaningful when MAP_PROVIDER == "mapbox"; pydeck ignores
# it for the "carto" fallback, so this line is safe either way.
# api_keys는 MAP_PROVIDER == "mapbox"일 때만 의미가 있다. "carto" 폴백일
# 때는 pydeck이 그냥 무시하므로 이 줄은 어느 경우든 안전하다.
deck = pdk.Deck(
    layers=[layer_blocks, layer_circle],
    initial_view_state=view_state,
    map_provider=MAP_PROVIDER,
    map_style=MAP_STYLE,
    api_keys={"mapbox": MAPBOX_TOKEN} if MAPBOX_TOKEN else None,
    tooltip={"text": "{region_nm}"},
)

deck.to_html(os.path.join(OUT_DIR, "hiroshima_3d_population.html"))

deck


### Lecture 4 summary / 4강 요약

**Key takeaways / 핵심 요약**
- The same area-weighting technique from Lecture 2 applies identically to a search-radius KPI -- the geometry changes (circle instead of difference), the fractional-overlap logic doesn't.
- A pre-filter built for one question (which areas are near a stop) is not automatically safe to reuse for a different question (which areas are in this circle) -- they need to be checked against the same set of candidates, not two different ones.

**핵심 요약**
- 2강의 면적가중 기법이 탐색반경 KPI에도 동일하게 적용된다 -- 지오메트리 연산은 바뀌어도(차집합 대신 원) 부분중첩 가중 로직은 같다.
- 한 질문(어느 지역이 정류장 근처인가)을 위해 만든 사전필터를 다른 질문(어느 지역이 이 원 안에 있는가)에 그대로 재사용하면 안전하지 않다 -- 같은 후보집합을 기준으로 검증해야지, 서로 다른 두 집합을 섞으면 안 된다.

**Real-world tip / 실무 팁**
This exact notebook had that second bug until this session's review: the 1,250m search circle was tested against `gdf_cand` (areas within 1,250m of *some stop city-wide*) instead of the full grid, so 3 real areas that genuinely overlapped the circle were silently dropped because their centroid happened to sit slightly outside the *other* filter. **Every time you reuse a filtered variable for a second purpose, ask what condition it was actually filtered by** -- if the new question's condition isn't the same condition, build the new answer from the unfiltered source instead.

**실무 팁**
바로 이 노트북이 이번 세션의 리뷰 전까지 그 두 번째 버그를 갖고 있었다: 1,250m 탐색원이 (도시 전역 아무 정류장에서나 1,250m 이내인 지역들의 집합) `gdf_cand`를 기준으로 검사되고 있어서, 실제로는 원과 겹치지만 중심점이 *다른* 필터 기준에서 살짝 벗어난 지역 3곳이 조용히 누락됐다. **필터링된 변수를 다른 목적으로 재사용할 때마다, 그게 애초에 어떤 조건으로 걸러진 건지 반드시 확인하자** -- 새 질문의 조건이 같지 않다면, 필터링 안 된 원본에서 새로 뽑아야 한다.

**Try it yourself / 직접 해보기**
Insert a new cell and try a smaller search radius around the same demo area, without touching `gdf_in`/`circle_6670`. How much does `total_pop` change between an 800m and 1,250m radius?

새 셀을 추가해서 `gdf_in`/`circle_6670`은 건드리지 않고, 같은 데모 지역에 더 작은 탐색 반경(800m)을 직접 시도해보자.

<details>
<summary>Show one way to do it / 정답 예시 보기</summary>

```python
my_radius_m = 800.0  # instead of RADIUS_M = 1250.0
my_circle = sel_center.buffer(my_radius_m)
my_inside = gdf_grid[gdf_grid.geometry.intersects(my_circle)].copy()
my_inside["in_frac"] = (my_inside.geometry.intersection(my_circle).area / my_inside.geometry.area).clip(0, 1)
print(f"total_pop at {my_radius_m:.0f}m: {(my_inside['pop'] * my_inside['in_frac']).sum():,.0f}")
```

</details>

### Closing / 마무리

이 발표의 주제는 히로시마의 숫자 자체가 아니라, **바이브 코딩으로 공간분석을 설계하고 데이터로 가설을 검증하는 방법**이다.

전체 흐름은 다음과 같다: 자연어 질문을 세우고 → 필요한 공간분석 기능을 선택하고 → 코드를 빠르게 실행하고 → 지도와 수치로 검증하고 → 인구·거리·네트워크 같은 맥락을 추가해 인사이트를 정교화한다. GeoPandas는 공간 데이터 모델과 조인을, Folium과 pydeck은 결과 전달을, OSMnx는 실제 이동 경로를 담당한다.

최종 결과는 “어느 곳이 취약하다”는 단정이 아니다. 어떤 데이터와 가정으로 그렇게 판단했는지 설명할 수 있고, 다른 도시·다른 데이터로 다시 실행할 수 있는 분석 패턴이다. 좋은 바이브 코딩은 코드를 대신 써주는 데서 끝나지 않고, 질문·검증·해석의 속도를 높인다.